# FRIDA per-category LoRA — V12 host-RAM-safe

Эта версия сделана под текущий реальный crash:

```text
SIGTERM from Kaggle parent
→ all TPU workers terminated
→ BrokenProcessPool in torch_xla.launch
```

Последний полезный лог перед падением показывал примерно `ru_maxrss=25 GB` на одном rank.
Это не похоже на обычную Python-ошибку модели и не похоже на HBM OOM: процесс убивается
снаружи. Поэтому V12 специально снижает **host RAM / process fan-out**.

## Что изменено

- **Нет полного pre-fork scan 12.38M × 256 token cache.** Большой `input_ids.bin`
  остаётся lazy `memmap`; длина и позиции `Category:` / `Attributes:` вычисляются только
  для item rows, реально попавших в batch.
- **`DataLoader(num_workers=0)`**. Восемь XLA ranks уже являются восемью процессами;
  больше не создаются ещё 16+ вложенных loader workers, наследующих огромные mappings.
- LLM split читается через **Polars streaming**, только нужные колонки, без 11M-row
  pandas frame с Python-строками.
- `row1/row2` хранятся как `int32`, category как `int16`.
- Перед fork: `gc.collect() → malloc_trim(0) → gc.freeze()`.
- После fork убраны full-heap `gc.collect()` из training path.
- Логи разделяют:
  `rss_current`, `rss_peak`, `MemAvailable`.
  Старый `ru_maxrss` был peak RSS, хотя в логе назывался просто `rss`.
- Quiet checkpoint каждые **15 минут**.
- Checkpoint теперь **adapter + head only**: optimizer state никогда не использовался
  resume-кодом, а его XLA→CPU serialization только создавал лишний RAM spike.

Архитектура FRIDA, LoRA/head, loss, full LLM train split, category order и LR schedule
не менялись.

## Важный момент про 9 часов

Текущая версия обучения использует примерно весь `11.09M` LLM train split.
При старой измеренной скорости порядка `170 rows/s` один полный проход — больше одной
9-часовой TPU session. Поэтому этот kernel по-прежнему рассчитан на **resume между
сессиями**. V12 уменьшает вероятность аварийного kill и резко сокращает потерю
прогресса при внешнем SIGTERM.

In [ ]:
# Write the patched training script into /kaggle/working.

from pathlib import Path

SCRIPT = Path("/kaggle/working/frida_percat_lora_v12_hostram_safe.py")

PATCHED_SOURCE = '# %% [code]\n"""\n\nV12 HOST-RAM-SAFE PATCH (2026-08-22)\n===========================================================================\nThe latest real crash was not a Python training exception. All XLA workers\nreceived SIGTERM from Kaggle\'s parent process and torch_xla only surfaced a\nBrokenProcessPool afterward. The last useful line reported rank1 ru_maxrss at\nabout 25 GB. This version attacks HOST RAM pressure, not HBM:\n\n  * Do NOT scan the 12.38M x 256 token memmap pre-fork. Lengths and\n    Category:/Attributes: positions are derived lazily only for rows touched by\n    a batch.\n  * DataLoader num_workers=0. Eight XLA ranks are already eight processes;\n    nested DataLoader workers inherited the same huge mappings.\n  * Load the 11M-row LLM split with Polars streaming and only needed columns,\n    rather than a pandas frame with Python strings.\n  * gc.collect()+malloc_trim(0) once before fork, then gc.freeze() to preserve\n    copy-on-write sharing; no full-heap gc.collect() inside child training.\n  * Log current RSS, peak RSS, and host MemAvailable separately. ru_maxrss is a\n    PEAK, not current RSS.\n  * Quiet checkpoint interval reduced from 2h to 15m; checkpoints are weights-only because resume never reloads optimizer.pt.\n\nModel architecture, objective, full LLM-train split, category order and LR\nschedule are otherwise unchanged.\n\nFRIDA per-category specialists: merge the general adapter into the base\nweights, then train 20 INDEPENDENT (LoRA + linear classifier) pairs, one per\ncategory.\n\nWHY per-category at all\n===========================================================================\nThe competition metric is a plain mean of per-category PR-AUC over 20\ncategories -- it is literally 20 independent problems averaged. A single\nmodel has to spend capacity reconciling categories whose positive rates\ndiffer by 7.7x (measured over the real data: 0.073 for the sparsest to 0.562\nfor the densest). Specialists do not.\n\nThe general model is NOT thrown away: the already-validated adapter (val\n0.7660) is MERGED into the base weights first, so every specialist starts\nfrom the strong general model and only has to learn its own category\'s\ndeviation from it.\n\nTHE MERGED WEIGHTS ARE SAVED BEFORE TRAINING STARTS\n===========================================================================\nmerge_and_save_base() runs pre-fork, on CPU, and writes the merged encoder\nto OUTPUT_DIR immediately -- before a single training step. Two reasons:\n(1) they are a deliverable in their own right (inference needs merged base +\none adapter per category), and (2) if training later dies, the merge\nsurvives in /kaggle/working, which Kaggle persists even for a cancelled or\nhard-capped session. Same "persist the expensive thing early" logic the rest\nof this project\'s checkpointing follows. Cost: ~1.65GB of the 20GB output\nbudget, plus ~20 x 25MB of per-category adapters = well inside it.\n\nSEQUENTIAL CATEGORIES, NOT ONE CATEGORY PER CORE\n===========================================================================\nThe tempting layout is 8 categories on 8 TPU cores at once, in waves of\n(8, 8, 4). It is the wrong shape here, for a reason that would have been\nsilent rather than loud:\n\n    xm.optimizer_step() ALL-REDUCES gradients across cores.\n\nWith a different LoRA per core that averages 8 different categories\'\ngradients into each other on every step, quietly destroying the very\nspecialization the design exists to create. Avoiding it means dropping to a\nbare optimizer.step(), which is exactly the call this project\'s\nkaggle-experiment-safety skill lists as a silent run-killer on TPU.\n\nSo: categories run ONE AT A TIME, each using all 8 cores in ordinary data\nparallelism. Same total compute (the per-category row counts sum to the full\ndataset), standard proven code path, and measurably LESS wall-clock: the\nwave scheme costs max-of-wave x 3 = ~59.1K pair-units against 45.7K for\nsequential-with-DP, because seven cores idle waiting on each wave\'s largest\ncategory.\n\nOne LoRA module is created ONCE and its weights/optimizer are RESET between\ncategories, rather than building a fresh PeftModel 20 times -- XLA compiles\nper shape, and the shape never changes, so this keeps the compile count at\none instead of twenty.\n\nTIME BUDGET (arithmetic, from this project\'s own measured 120 rows/s at\nPAIR_MAX_LENGTH=256 -- see frida-llm-tpu-sft\'s docstring)\n===========================================================================\nPAIR_MAX_LENGTH=192 scales the attention term by (192/256)^2 and the rest\nlinearly, giving ~170 rows/s. Per category: all its human train rows\n(~14.6K) + LLM_ROWS_PER_CATEGORY (100K) = ~114.6K rows; x20 categories =\n~2.36M rows = ~3.9h. Comfortably inside one 9h TPU session with the 8.5h\nstop, with quota left for a second pass.\n\nSPLIT: THE OLD CANONICAL ONE, DELIBERATELY\n===========================================================================\nThis kernel uses the 80/20 pair-level seed-42 split, NOT the newer\nitem-disjoint 70/30 one (data/splits/matches_item_disjoint_70_30_seed42.*).\nThat is not an oversight. The adapter being merged here was trained and\nvalidated against the old split; re-splitting under it would move items it\nalready trained on into the new val half and silently inflate every\nper-category number. A checkpoint keeps the split it was born with.\n\nWHAT IS WEIGHTED, AND WHY IT IS NOT CATEGORY SIZE\n===========================================================================\nCategory SIZES are nearly uniform in this dataset -- 17,286 to 23,357 pairs,\na 1.4x spread -- so weighting by inverse category size would be close to a\nno-op. Positive RATE is what varies (7.7x). Each specialist therefore gets\nits OWN pos_weight, computed from its own training mix, plus a 2x weight on\nhuman-labeled rows relative to LLM soft-labeled ones.\n\nHEAD SCHEDULE, GRADIENT CLIPPING, FAIL-LOUD ON NaN\n===========================================================================\nThe classifier is randomly initialised while the backbone is already strong,\nso the two need different schedules. Two param groups: the head ramps to\nHEAD_LR (3x the adapter\'s) over a SHORT warmup (HEAD_WARMUP_STEPS), holds\nthat peak flat until the adapter\'s own (longer) warmup ends, then both\ncosine-decay together to a floor.\n\nThis is a real fix, not a first-principles design choice: an earlier\nversion jumped the head straight to peak at step 0 with no ramp and no\ngradient clipping anywhere, and a real run (frida-percat-lora.log)\nreproducibly hit loss=nan by the very first logged step on 15/20\ncategories, with 2 more drifting into NaN once a harder batch hit later --\n17/20 specialists lost, only 3 (07/10/14) survived. Fixed with three\nchanges together, not any one alone: the short head ramp above,\nHEAD_LR_MULTIPLIER backed off 5.0->3.0, and grad-norm clipping\n(GRAD_CLIP_NORM) before every optimizer step. A NaN loss also now fails\nLOUDLY and immediately (TrainingDivergedError, raised the moment\ntorch.isfinite(loss) is False) instead of silently continuing to "train"\non garbage for the rest of that category\'s steps -- the previous behavior\nwasted each failing category\'s full step budget before the problem\nsurfaced at eval time.\n\nV6: RESUME FROM v5, FULL LLM SPLIT, QUIET CHECKPOINTS\n===========================================================================\nThree changes, all per direct instruction, none of them touching the NaN\nfix above:\n\n  1. RESUME, NOT SKIP. v5 treated an already-good external checkpoint as\n     "done, copy it forward" (07/10/14 in that real run). v6 treats ANY\n     existing checkpoint -- a validated _best, or failing that a quiet\n     mid-training _last -- as a starting point to CONTINUE training from,\n     via load_specialist_from_checkpoint_() (probe-delta verified, same\n     no-op-detection reasoning as merge_and_save_base\'s own check). A\n     category with no checkpoint anywhere still falls back to\n     reset_specialist_\'s fresh init. Resumed categories train at\n     LR * LR_RESUME_MULTIPLIER (0.5) -- a smaller step for continuing an\n     adapter that is already at a good point, same reasoning this project\'s\n     own LLM continue-SFT already used (LR halved from its from-scratch\n     peak). The warmup/clip shape is UNCHANGED for resumed categories, only\n     the peak magnitude scales down.\n  2. FULL LLM TRAIN SPLIT, NOT A SUBSAMPLE. v5 capped LLM rows at\n     LLM_ROWS_PER_CATEGORY (100K/category) via stratified_llm_sample. v6\n     loads frida-llm-train-val-split-no-leak-retroactive v2\'s own output and\n     uses every row marked split=="train" (~11.09M rows, category-balance\n     and target-level stratification already applied by that kernel, not\n     re-applied here) -- no per-category cap. That kernel\'s own "train"\n     covers everything NOT held out for its fixed, unchanged 100K val; its\n     original 1,000,000-row reproduction (the exact rows v5 already trained\n     specialists on) is a verified subset, so nothing already trained on\n     moves into a held-out set under this run.\n  3. QUIET CHECKPOINTS, ACTUALLY WIRED UP. CHECKPOINT_EVERY_SECONDS existed\n     in v5 but nothing ever checked it -- a killed session lost the WHOLE\n     in-progress category, not just the time since its last checkpoint. v6\n     checks wall-clock against it (7200s = 2h) inside the per-step loop and\n     saves adapter+head under f"{label_prefix}_last" (val_pr_auc explicit\n     None, the same unvalidated-checkpoint convention save_checkpoint\n     already uses) when crossed -- no eval, so this is cheap enough to run\n     mid-category without disrupting the training loop\'s own timing.\n\nREAL CRASH: load_specialist_from_checkpoint_\'s missing-keys check was itself\nwrong on its first real run (all 8 ranks died identically at cat00, ~6.3min\nin). set_peft_model_state_dict() does model.load_state_dict(adapter_sd,\nstrict=False) under the hood, and adapter_sd only ever holds the LoRA delta\nweights -- every base-model weight (base_model.model.*, *.base_layer.weight,\nrelative_attention_bias, layer_norm, ...) is CORRECTLY reported "missing" by\nstrict=False on every single load. The real traceback showed exactly that:\n~230 "missing" keys, every one a base weight, zero lora_ keys -- the adapter\nitself loaded fine, the check was just wrong. Fixed by filtering missing_keys\nto only lora_-containing entries before deciding it\'s an error; a missing\nbase weight is how a correct partial adapter load is supposed to look.\n\nREAL CRASH #2: fixing #1 let the resume actually happen -- and the very next\nrun crashed the WHOLE process at cat00\'s first training step (loss=nan at\nstep 1/339), taking all 8 ranks and every other category down with it. Two\ncompounding problems, both fixed together:\n  1. cat00_last (and every OTHER non-_best checkpoint in\n     frida-with-merged-lora-and-8-10-14-cat-loras) is a casualty of the\n     ORIGINAL pre-NaN-fix run this project already diagnosed once (see the\n     NaN root-cause section above) -- that dataset\'s name ("...8-10-14...")\n     literally IS the user\'s own recollection of which 3 categories survived\n     it, and cat07_best/cat10_best/cat14_best are the only _best entries\n     present. Its 17 _last checkpoints are pre-fix weights already pushed\n     into NaN, not legitimately-cut-short good ones. load_specialist_from_\n     checkpoint_ now checks torch.isfinite() on every loaded lora_ and head\n     param right after loading and raises if any are non-finite, so a\n     corrupted checkpoint is caught before a single training step touches it.\n  2. Independently of #1: TrainingDivergedError was NEVER actually caught at\n     "the category level" despite the class\'s own docstring and the step-\n     loop\'s own comment both claiming it was -- the only try/except at that\n     scope wrapped just the evaluate() call, well outside the epoch loop\n     TrainingDivergedError is raised from, so it propagated straight out of\n     _mp_fn and killed the whole run (confirmed by the real traceback: line\n     numbers pointing at the raise inside _mp_fn itself, uncaught). Fixed by\n     wrapping the per-category epoch loop in its own try/except\n     TrainingDivergedError: on catch, save a quiet _last checkpoint (marked\n     "diverged": true), record results[cat_name] = None, and move on to the\n     next category -- exactly the behavior the old comment already claimed\n     existed. The resume call site also now catches any\n     load_specialist_from_checkpoint_ failure (not just non-finite weights)\n     and falls back to reset_specialist_ rather than propagating -- a\n     category that can\'t be resumed trains from scratch instead of taking\n     the whole run down.\n\nREAL CRASH #3 (host RAM OOM, 2026-08-22) -- fixing #2 exposed a resource\nleak that used to be structurally impossible to hit\n===========================================================================\nThe very next real run (with #2\'s fix applied) survived the whole 20\n-category loop crashing outright, but died ~1500s / 10 categories in with\nKaggle\'s own "tried to allocate more memory than is available" -- host RAM,\nnot HBM. Root cause: `del mp_loader, train_loader, sampler` only ever sat\nAFTER the step loop (a normal epoch end reaches it); a TrainingDivergedError\nraised mid-step jumps straight past it to the except clause, which had no\ncleanup of its own. val_full_loader/val_remainder_loader (also\nnum_workers=2 each) were never explicitly freed AT ALL, relying only on the\nimplicit refcount drop when those names get reassigned next category.\nBefore v6 this never mattered: divergence used to crash the WHOLE process\nimmediately (nothing had a chance to accumulate), and v5\'s own\n"already-good, skip via continue" path returned before eval loaders were\neven constructed. v6 made both of those the exception rather than the rule\n-- resuming means every category (not just the ones needing real training)\nbuilds eval loaders, and catching divergence means a category that fails in\n2-5 steps immediately starts the next one. The real run that found this hit\n8/10 diverging categories in ~1500s, i.e. many rapid DataLoader create\n-then-abandon cycles across 8 ranks in parallel -- exactly the shape that\nturns "technically a leak" into "exhausts host RAM in 25 minutes." Fixed by\nadding the same del+gc.collect() to the except-TrainingDivergedError branch,\nand an unconditional del of both val loaders once they\'re no longer needed\nfor the category (moved from "never" to "every category, right after use").\n\nTwo more additions, same session, aimed at the root cause this exposed\nrather than another symptom:\n  - SMOKE TEST (_smoke_test_train_step/_smoke_test_eval_step/\n    _smoke_test_real_loader_batch, run once before the category loop):\n    frida-percat-lora never had one, unlike frida-full-sft (which earned\n    its own the hard way, five real crashes). Every bug in THIS kernel\'s\n    history -- three separate checkpoint-path bugs, the over-strict\n    missing-keys check, this OOM -- was found only after a real, paid TPU\n    run. Doesn\'t fix any of those; means the next HBM-fit/shape bug costs\n    1-2 minutes instead of hours.\n  - MAX_CONSECUTIVE_CATEGORY_FAILURES=6: the SAME real run that found the\n    OOM also showed 8 of its first 10 categories diverging to NaN from a\n    FRESH init, using the exact HEAD_LR_MULTIPLIER/warmup/GRAD_CLIP_NORM\n    config already described above as "the NaN fix" -- across multiple\n    independent real runs now (this one and the original one that produced\n    frida-with-merged-lora-and-8-10-14-cat-loras), that config has never\n    actually gotten past ~3/20 categories surviving. That is a separate,\n    deeper, still-unsolved problem (see frida-percat-lora-nan-diagnose) --\n    but whatever its cause, if it is systemic (bad merged base, bad data\n    mix, a config regression) rather than per-category, burning the full\n    8.5h MAX_TRAIN_SECONDS to discover that one category at a time, 20\n    times, is pure waste. Six consecutive category-level failures now stops\n    the run loudly with a clear message instead.\n\nREAL CRASH #3, ROUND 2: the leak fix above did NOT fully solve the OOM --\nsame crash, one category count earlier\n===========================================================================\nThe very next real run (leak fix + smoke test + circuit breaker all\napplied) hit the SAME "tried to allocate more memory than is available"\nkill, this time starting category 04 at ~836s instead of surviving to\ncategory 09 at ~1479s in the run that found the leak. The per-category\ndivergence steps were IDENTICAL to the previous run (00->step40, 01->step2,\n02->step5, 03->step32 -- confirming the NaN divergence itself is\ndeterministic, not noise), but each category\'s own wall-clock grew 55-96s\npast the leak-fix-free baseline (631s/686s/759s/834s vs 552s/605s/668s/738s)\n-- a GROWING gap, not a fixed one-time offset the smoke test alone would\nproduce. Not fully root-caused, but a real, checkable hypothesis: the fix\nadded THREE gc.collect() calls per diverged category (step-level,\ncategory-level, val-loader) where one used to run; CPython\'s cyclic\ncollector walks the whole object graph, and under fork+refcount semantics,\nmerely VISITING an object bumps its refcount field -- which lives on the\nsame memory page as the object -- triggering copy-on-write duplication of\nthat page in this rank\'s process even for otherwise-untouched, pre-fork\n-shared data (the large item-token caches, val_by_cat/train_by_cat). More\ncollect() calls could mean more incremental COW duplication of supposedly\n-shared memory, which would show up exactly as "gets slower and eventually\nOOMs, worse than before the extra collect() calls" -- what was observed.\nTwo changes, aimed at making this checkable rather than guessed a third\ntime:\n  1. Consolidated three gc.collect() calls per diverged category down to\n     one (kept the del statements -- those alone already drop non-cyclic\n     objects to refcount 0 immediately, no collect() needed for that part;\n     collect() only adds value for reference cycles, and one pass per\n     category catches those same as three would).\n  2. Added direct RSS logging (resource.getrusage) to the per-category CAT\n     log line -- the next real run will show the actual memory curve\n     instead of this docstring inferring it from wall-clock gaps a third\n     time.\n\nThe RSS logs from that run then SETTLED the question the gc.collect()\nguess was trying to answer, and pointed somewhere else entirely: RSS did\nNOT climb category-over-category (17.9-18.9GB at cat00, then a ONE-TIME\njump to 19.6-22.9GB at cat01, then FLAT through cat02/03/04) -- not a\nleak, a high BASELINE. 8 ranks x ~21GB =~ 168GB, evidently close to this\nTPU v5e-8 VM\'s actual host RAM ceiling regardless of category content.\n\nV11: build_specialist_model_cpu + MpModelWrapper -- fixing the baseline\nitself, not another symptom of hitting it\n===========================================================================\nPyTorch/XLA\'s own docs describe the exact anti-pattern this kernel had:\n"instead of creating models on each multiprocessing process ..., the model\nis created once at global scope, and then moved into each device inside\nthe spawn() target function" (xmp.MpModelWrapper). This kernel\'s own\n_load_shared_data() already does this correctly for DATA (loaded once in\nmain(), pre-fork) -- but the MODEL was built inside _mp_fn, i.e. every one\nof 8 ranks independently called T5EncoderModel.from_pretrained() (a real\n823M-param disk read + fresh CPU allocation) and get_peft_model() AFTER\nthe fork, none of it shared. build_specialist_model_cpu now does exactly\nthat construction ONCE in main(), pre-fork; the result is wrapped in\nxmp.MpModelWrapper and every rank\'s own _mp_fn just does\n`_WRAPPED_MODEL.to(device)` -- a cheap transfer of already-loaded host\npages, not a fresh load. Per-category reset_specialist_/\nload_specialist_from_checkpoint_ are unaffected: they mutate the\nalready-on-device, per-rank-independent tensors .to(device) produces, same\nas before.\n\nPushed deliberately WITHOUT also changing init_lora_weights or batch size\n(that combination went to a separate account/push instead, see\nkernel-metadata.json history) -- two independent fixes for two independent\nproblems (host RAM baseline vs. NaN), run in parallel on separate quotas\nspecifically so which one actually helps is separable, not guessed a\nfourth time from a run that changed everything at once.\n\nINHERITED LESSONS (kept verbatim from frida-llm-tpu-sft, do not re-derive)\n===========================================================================\n  1. autocast INSIDE the model\'s own forward(), never around the outer call.\n  2. PeftModel.from_pretrained(..., is_trainable=True).\n  3. Fixed batch sizes, NEVER a TPU probe -- each probed candidate compiles\n     its own XLA program and those stay resident in HBM.\n  4. xm.optimizer_step(optimizer), not optimizer.step().\n  5. drop_last=True on every loader so XLA only ever sees two shapes.\n  6. del logits, loss before continue on a skipped step.\n  7. ALL CPU-only setup ONCE in main() before torch_xla.launch().\n  8. Results mirrored to JSON in /kaggle/working -- a cancelled session can\n     come back with a 0-byte log.\n"""\nimport os\nimport sys\nimport gc\nimport json\nimport math\nimport resource\nimport time\nimport hashlib\nimport shutil\nimport subprocess\nimport ctypes\n\nsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "polars", "peft"])\n\nimport numpy as np\nimport pandas as pd\nimport polars as pl\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.utils.data.distributed import DistributedSampler\n\nfrom transformers import T5EncoderModel, set_seed\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import average_precision_score\n\ntry:\n    import peft.import_utils\n    import peft.tuners.lora.torchao\n    peft.import_utils.is_torchao_available = lambda: False\n    peft.tuners.lora.torchao.is_torchao_available = lambda: False\nexcept Exception as e:\n    print(f"[WARN] torchao patch failed (likely fine): {e}")\n\nfrom peft import (LoraConfig, PeftModel, TaskType, get_peft_model,\n                   get_peft_model_state_dict, set_peft_model_state_dict)\nfrom safetensors.torch import save_file as safetensors_save_file\nfrom safetensors.torch import load_file as safetensors_load_file\n\nimport torch_xla\nimport torch_xla.core.xla_model as xm\nfrom torch_xla import runtime as xr\nimport torch_xla.distributed.parallel_loader as pl_xla\nimport torch_xla.distributed.xla_multiprocessing as xmp\n\n\n# =====================================================================\n# CONFIG\n# =====================================================================\n\nSEED = 42\nMODEL_NAME = "ai-forever/FRIDA"\n\nMATCHES_PATH_HINT = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/matches.parquet"\nMATCHES_LLM_PATH_HINT = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/matches_llm.parquet"\n# v2 of frida-llm-train-val-split-no-leak-retroactive: train = every row\n# NOT held out for the fixed 100K val (~11.09M rows), not a subsample --\n# see that kernel\'s own docstring. Publishes to the SAME dataset id\n# frida-full-sft already references (maxamsecret/frida-llm-train-val-split),\n# just a new version of it.\nLLM_SPLIT_PATH_HINT = (\n    "/kaggle/input/datasets/maxamsecret/"\n    "frida-llm-train-val-split/llm_train_val_split.parquet"\n)\nITEMS_HUMAN_PATH_HINT = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/items_human.parquet"\n\n# v2 caches (frida-tokenize-items-cpu): correct char budget (140/32/529,\n# see reports/experiment_log.md "Char-budget rule v2") + the light v2\n# attribute cleanup. NOT the v1 caches, which carry the under-scaled\n# 128/64/384 budget sized for MiniLM\'s old 128-token regime.\n#\n# Published as ONE dataset covering both scopes (frida-tokenize-items-cpu\'s\n# own two publish_dataset() calls failed at the very last step -- the\n# tokenization itself completed correctly, confirmed via cache_meta.json:\n# 711,304 human items, v2 policy applied to 711,232 of them -- so the user\n# published /kaggle/working by hand as a single dataset instead). These\n# hints therefore won\'t resolve to an exact path; that\'s fine,\n# _find_v1_cache_root() falls back to a content-matching walk of\n# /kaggle/input keyed on n_items, which is what actually finds them.\nHUMAN_CACHE_HINT = "/kaggle/input/datasets/maxamsecret/frida-tokenize-items-v3/human_scope"\nLLM_CACHE_HINT = "/kaggle/input/datasets/maxamsecret/frida-tokenize-items-v3/llm_scope"\n\nITEMS_FULL_PATH_HINT = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/items.parquet"\n\nITEM_MAX_LENGTH = 256\n# 192, not 256: this is the length the real judging stand can actually\n# SERVE inside its time budget (176 passed at 0.4087, 192 measured at\n# 1.65x over Public\'s budget and so is the current ceiling to aim at),\n# and training at the serving length removes the train/serve mismatch\n# that every FRIDA submission so far has paid for.\nPAIR_MAX_LENGTH = 192\n\nMAX_CATEGORY_CHARS = 64\n\nCATEGORY_LABEL_TOKEN = 55917\nATTRIBUTES_LABEL_TOKEN = 91837\nLABEL_COLON_TOKEN = 30\n\n# A single nn.Linear per category, per the design: 20 small specialists,\n# not 20 copies of the multi-layer head. The merged backbone already\n# carries the general model\'s capacity; the head only has to place one\n# category\'s decision boundary.\nHEAD_DROPOUT = 0.1\n\n# Deliberately SMALLER and more regularized than the project default\n# (r=16/alpha=32/dropout=0.10): each specialist sees ~115K rows, not\n# 11M, so a larger adapter would mostly memorize.\nLORA_R = 8\nLORA_ALPHA = 16\nLORA_DROPOUT = 0.15\nLORA_TARGET_MODULES = ["q", "k", "v", "o", "wi_0", "wi_1", "wo"]\n\n# BERTA recipe\'s own numbers, and the MiniLM continue-SFT fix\'s numbers.\nLR = 2e-4                  # adapter peak\n# v6: categories resuming from an already-trained checkpoint (see\n# load_specialist_from_checkpoint_) use LR * LR_RESUME_MULTIPLIER as their\n# peak instead of the full fresh-init LR -- continuing an adapter that is\n# already at a good point should take smaller steps than initializing one\n# from a random start, same reasoning any continued-fine-tuning run in\n# this project has already used (e.g. the original LLM continue-SFT\'s own\n# LR=2e-4, half of a from-scratch run\'s peak). The warmup/clip machinery\n# below is UNCHANGED for resumed categories -- only the peak magnitude\n# scales down, not the shape of the ramp.\nLR_RESUME_MULTIPLIER = 0.5\n\n# 5.0 combined with an instant step-0 jump to peak (see\n# _make_head_lr_lambda) is the confirmed root cause of a reproducible\n# early-NaN failure: a real run (frida-percat-lora.log) showed loss=nan\n# already by the FIRST logged step on 15/20 categories, with the other 2\n# of the 17 failures drifting into NaN a bit later once a harder batch\n# hit -- i.e. a freshly-initialised linear head jumping straight to a\n# 5x-multiplied LR with no ramp and no gradient clipping anywhere in the\n# step. Backed off to 3.0 and given its own short ramp, plus clipping\n# added to the step itself (see the training loop) -- not relying on\n# either fix alone.\nHEAD_LR_MULTIPLIER = 3.0\nHEAD_WARMUP_STEPS = 20     # short ramp 0 -> peak, then flat until warmup_steps\nGRAD_CLIP_NORM = 1.0\nWEIGHT_DECAY = 0.01\nWARMUP_RATIO = 0.10\nHUMAN_WEIGHT = 2.0         # human rows count double against LLM soft labels\n\n# Scaled from a PROVEN point, not from a probe (probing is forbidden on\n# TPU -- see docstring lesson 3). frida-tpu-human-sft runs 16/32 at\n# PAIR_MAX_LENGTH=256 without OOM. Per-sample activation memory at 192 is\n# 0.62x of that at 256 (attention scales with seq^2, the rest linearly),\n# which would permit ~26; 24 keeps the same margin the proven point had.\nPER_CORE_TRAIN_BATCH = 24\nPER_CORE_EVAL_BATCH = 48\n\n# v5-only: capped LLM rows/category for the from-scratch run. v6 trains on\n# the full frida-llm-train-val-split-no-leak-retroactive train split instead\n# (see LLM_SPLIT_PATH_HINT below, no cap) -- this constant is kept only\n# because stratified_llm_sample (still accurate documentation of the target-\n# level distribution) takes it as a parameter and nothing currently calls\n# that function from _load_shared_data any more.\nLLM_ROWS_PER_CATEGORY = 100_000\nEPOCHS = 1                     # per category\nMAX_TRAIN_SECONDS = 30600      # 8.5h. Kaggle\'s TPU session cap is 9h (the 12h\n                                # cap is GPU) -- raise this only if a real TPU\n                                # run here is observed surviving past 9h.\n# v6: 3600 -> 7200, per direct instruction. Quiet checkpoints now save the\n# FULL state (adapter + head, see the per-step check below) so a session\n# killed mid-category loses at most 2h of that category\'s progress instead\n# of losing the whole category.\nCHECKPOINT_EVERY_SECONDS = 900\nLOG_EVERY_STEPS = 50\n\n# Used ONLY to size the LR schedule (see docstring). Measured on\n# frida-tpu-human-sft at this exact model/sequence/batch. If a future run\n# reports a different rate, update this rather than the schedule shape.\nPROJECTED_ROWS_PER_SEC = 170   # 120 measured at seq=256, scaled to seq=192\nLR_MIN_RATIO = 0.05            # LR floor as a fraction of peak\n\nWORKING_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."\nOUTPUT_DIR = os.path.join(WORKING_DIR, "frida_percat_lora")\nMERGED_BASE_DIR = os.path.join(OUTPUT_DIR, "merged_base")\nRESULTS_JSON = os.path.join(OUTPUT_DIR, "percat_results.json")\n\nt_script_start = time.perf_counter()\n\n\ndef _elapsed():\n    return time.perf_counter() - t_script_start\n\n\ndef _log(rank, stage, msg, force=False):\n    if rank == 0 or force:\n        print(f"[{_elapsed():8.1f}s][rank{rank}] {stage}: {msg}", flush=True)\n\n\ndef _current_rss_mb():\n    try:\n        with open("/proc/self/status") as f:\n            for line in f:\n                if line.startswith("VmRSS:"):\n                    return float(line.split()[1]) / 1024.0\n    except Exception:\n        pass\n    return float("nan")\n\n\ndef _peak_rss_mb():\n    # Linux ru_maxrss is a PEAK, not current RSS.\n    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0\n\n\ndef _mem_available_mb():\n    try:\n        with open("/proc/meminfo") as f:\n            for line in f:\n                if line.startswith("MemAvailable:"):\n                    return float(line.split()[1]) / 1024.0\n    except Exception:\n        pass\n    return float("nan")\n\n\ndef _malloc_trim():\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except Exception as e:\n        print(f"[MEM] malloc_trim unavailable: {type(e).__name__}: {e}", flush=True)\n\n\ndef _mem_log(prefix):\n    print(\n        f"[{_elapsed():8.1f}s][MEM] {prefix}: "\n        f"rss_current={_current_rss_mb():.0f}MB "\n        f"rss_peak={_peak_rss_mb():.0f}MB "\n        f"mem_available={_mem_available_mb():.0f}MB",\n        flush=True,\n    )\n\n\ndef _find_dataset_file(hint, filename):\n    if os.path.isfile(hint):\n        return hint\n    if os.path.isdir("/kaggle/input"):\n        for root, _dirs, files in os.walk("/kaggle/input"):\n            if filename in files:\n                return os.path.join(root, filename)\n    return hint\n\n\ndef _find_resume_adapter_dir(prefer_name="best"):\n    """Collects EVERY directory under /kaggle/input containing an\n    adapter_config.json, rather than trusting os.walk\'s first hit.\n\n    Real bug #1, found the hard way: the previous version hardcoded a\n    priority path with a typo -- "/kaggle/inputdatasets/..." instead of\n    "/kaggle/input/datasets/..." (missing slash) -- which never exists,\n    so it silently fell through every time to a bare os.walk("/kaggle/\n    input") first-match. With TWO valid adapters now mounted (this SFT\n    checkpoint\'s own "best", and the older epoch4 checkpoint), which one\n    won was undefined -- the run this exact bug shipped in got lucky, a\n    future one might not.\n\n    Real bug #2, found the hard way when frida-with-merged-lora-and-8-10-\n    14-cat-loras (20 per-category checkpoints) was mounted alongside the\n    general SFT checkpoints: `candidates` holds the directory that\n    DIRECTLY contains adapter_config.json. Under this project\'s own\n    save_checkpoint (and frida-llm-tpu-sft\'s own checkpointing), that\n    directory is ALWAYS literally named "adapter" -- the checkpoint\'s real\n    label ("best", "checkpoint_hour05", "cat07_best", ...) is one level up,\n    in the PARENT directory\'s name. Comparing prefer_name against the\n    candidate\'s OWN basename can therefore never match anything saved this\n    way -- a real run hit exactly this, 0/29 candidates matched "best",\n    RuntimeError. Fixed by preferring the parent\'s name when the candidate\n    itself is named "adapter", falling back to the candidate\'s own name\n    for any adapter saved without that extra nesting."""\n    if not os.path.isdir("/kaggle/input"):\n        return None\n    candidates = []\n    for root, _dirs, files in os.walk("/kaggle/input"):\n        if "adapter_config.json" in files:\n            candidates.append(root)\n    if not candidates:\n        return None\n\n    def _label(c):\n        c = c.rstrip("/\\\\")\n        base = os.path.basename(c)\n        return os.path.basename(os.path.dirname(c)) if base == "adapter" else base\n\n    preferred = [c for c in candidates if _label(c) == prefer_name]\n    if len(candidates) > 1:\n        print(f"[SETUP] {len(candidates)} candidate adapter dirs under /kaggle/input: "\n              f"{candidates}", flush=True)\n        if len(preferred) != 1:\n            raise RuntimeError(\n                f"multiple adapter checkpoints present but exactly one named "\n                f"\'{prefer_name}\' was expected (found {len(preferred)}) -- refusing to "\n                f"guess which to resume. Candidates: {candidates}")\n        print(f"[SETUP] using the one named \'{prefer_name}\': {preferred[0]}", flush=True)\n    return preferred[0] if preferred else candidates[0]\n\n\ndef _pattern_starts_chunked(input_ids, n_items, tok_a, tok_b, chunk=500_000):\n    """Chunked so the boolean mask never materialises at full 12.38M x 255\n    scale at once -- that alone would be several GB of transient RAM on top\n    of the 12.7GB memmap, on the host that also has to survive an 8-way\n    fork."""\n    starts = np.full(n_items, -1, dtype=np.int32)\n    for lo in range(0, n_items, chunk):\n        hi = min(lo + chunk, n_items)\n        block = np.asarray(input_ids[lo:hi])\n        mask = (block[:, :-1] == tok_a) & (block[:, 1:] == tok_b)\n        rows, cols = np.where(mask)\n        if len(rows):\n            uniq_rows, first_idx = np.unique(rows, return_index=True)\n            starts[lo + uniq_rows] = cols[first_idx]\n        del block, mask, rows, cols\n    return starts\n\n\ndef _lengths_chunked(attn, n_items, chunk=500_000):\n    lengths = np.empty(n_items, dtype=np.int32)\n    for lo in range(0, n_items, chunk):\n        hi = min(lo + chunk, n_items)\n        lengths[lo:hi] = np.asarray(attn[lo:hi]).sum(axis=1)\n    return lengths\n\n\n# Two v1 caches are mounted at once and both look identical to a filename\n# scan, so the fallback search must disambiguate by CONTENT (n_items), not by\n# whichever directory os.walk happens to reach first. Silently training the\n# human split against the LLM cache would produce garbage that no assertion\n# downstream would catch.\nEXPECTED_N_ITEMS = {"human": 711_304, "llm": 12_384_610}\n\n\ndef _find_v1_cache_root(hint, expect_n_items):\n    for candidate in (hint,):\n        meta_path = os.path.join(candidate, "cache_meta.json")\n        if os.path.isdir(candidate) and os.path.exists(meta_path):\n            with open(meta_path) as f:\n                if json.load(f).get("n_items") == expect_n_items:\n                    return candidate\n    if os.path.isdir("/kaggle/input"):\n        for root, _dirs, files in os.walk("/kaggle/input"):\n            if "cache_meta.json" in files and "input_ids.bin" in files:\n                with open(os.path.join(root, "cache_meta.json")) as f:\n                    meta = json.load(f)\n                if meta.get("n_items") == expect_n_items:\n                    print(f"[CACHE] hint {hint} missed -- matched n_items={expect_n_items:,} "\n                          f"at {root}", flush=True)\n                    return root\n    raise RuntimeError(f"[CACHE] no v1 item cache with n_items={expect_n_items:,} found "\n                       f"(hint was {hint})")\n\n\ndef _derive_pad_id_lazy(input_ids, attn, n_items, max_length, name):\n    """Derive pad id from a tiny sample; never scan the full token cache."""\n    rng = np.random.RandomState(SEED + (0 if name == "human" else 1))\n    sample = rng.choice(n_items, size=min(2000, n_items), replace=False)\n    pads = set()\n    checked = 0\n    for r in sample:\n        r = int(r)\n        L = int(np.asarray(attn[r]).sum())\n        if 0 < L < max_length:\n            pads.add(int(input_ids[r, L]))\n            checked += 1\n            if len(pads) > 1:\n                break\n    if not checked:\n        raise RuntimeError(\n            f"[CACHE:{name}] no padded row found in lazy sample; cannot derive pad_id"\n        )\n    if len(pads) != 1:\n        raise RuntimeError(f"[CACHE:{name}] inconsistent pad token: {sorted(pads)}")\n    pad = pads.pop()\n    print(f"[CACHE:{name}] pad_id={pad} (lazy sample, no full-cache scan)", flush=True)\n    return pad\n\n\ndef load_v1_cache(hint, name):\n    root = _find_v1_cache_root(hint, EXPECTED_N_ITEMS[name])\n    with open(os.path.join(root, "cache_meta.json")) as f:\n        meta = json.load(f)\n\n    n_items, max_length = meta["n_items"], meta["max_length"]\n    print(\n        f"[CACHE:{name}] {root}: n_items={n_items:,} max_length={max_length} "\n        f"prefix={meta.get(\'prefix\')!r}",\n        flush=True,\n    )\n    if max_length != ITEM_MAX_LENGTH:\n        raise RuntimeError(\n            f"[CACHE:{name}] max_length={max_length} != {ITEM_MAX_LENGTH}"\n        )\n\n    ids_path = os.path.join(root, "ids.bin")\n    input_ids_path = os.path.join(root, "input_ids.bin")\n    attn_path = os.path.join(root, "attention_mask.bin")\n\n    for fname, want in (\n        (ids_path, n_items * 8),\n        (input_ids_path, n_items * max_length * 4),\n        (attn_path, n_items * max_length),\n    ):\n        got = os.path.getsize(fname)\n        if got != want:\n            raise RuntimeError(\n                f"[CACHE:{name}] {fname} size mismatch: got {got:,}, want {want:,}"\n            )\n\n    # V12: map, but do not sweep through, the huge token cache before fork.\n    ids = np.memmap(ids_path, dtype=np.int64, mode="r", shape=(n_items,))\n    input_ids = np.memmap(\n        input_ids_path, dtype=np.int32, mode="r", shape=(n_items, max_length)\n    )\n    attn = np.memmap(\n        attn_path, dtype=np.int8, mode="r", shape=(n_items, max_length)\n    )\n\n    # Only the id vector (~100 MB for the LLM cache) is intentionally touched.\n    order = np.argsort(ids, kind="stable").astype(np.int32, copy=False)\n    sorted_ids = np.asarray(ids[order])\n\n    pad_id = _derive_pad_id_lazy(input_ids, attn, n_items, max_length, name)\n\n    # Small integrity sample only.\n    sample_rows = np.random.RandomState(SEED).choice(\n        n_items, size=min(50, n_items), replace=False\n    )\n    for r in sample_rows:\n        r = int(r)\n        L = int(np.asarray(attn[r]).sum())\n        if L <= 0:\n            raise RuntimeError(f"[CACHE:{name}] row {r}: zero attention length")\n        if int(input_ids[r, 0]) != 1:\n            raise RuntimeError(\n                f"[CACHE:{name}] row {r}: expected bos=1, got {input_ids[r, 0]}"\n            )\n        if int(input_ids[r, L - 1]) != 2:\n            raise RuntimeError(\n                f"[CACHE:{name}] row {r}: expected eos=2, got {input_ids[r, L - 1]}"\n            )\n\n    return dict(\n        name=name,\n        n_items=n_items,\n        max_length=max_length,\n        input_ids=input_ids,\n        attention_mask=attn,\n        order=order,\n        sorted_ids=sorted_ids,\n        pad_id=pad_id,\n    )\n\n\ndef lookup_rows(cache, item_ids):\n    """Vectorised id -> cache row. Returns -1 where an id is absent, so the\n    caller can fail loudly at setup instead of silently embedding whatever\n    sits at row 0."""\n    pos = np.searchsorted(cache["sorted_ids"], item_ids)\n    pos_clipped = np.clip(pos, 0, len(cache["sorted_ids"]) - 1)\n    hit = cache["sorted_ids"][pos_clipped] == item_ids\n    rows = np.where(hit, cache["order"][pos_clipped], -1)\n    return rows.astype(np.int32, copy=False)\n\n\n# =====================================================================\n# PAIR ASSEMBLY (byte-identical to every other FRIDA kernel)\n# =====================================================================\n\ndef allocate_pair_budget(len1, len2, budget):\n    half = budget // 2\n    if len1 <= half and len2 <= half:\n        return len1, len2\n    if len1 <= half:\n        return len1, budget - len1\n    if len2 <= half:\n        return budget - len2, len2\n    return half, budget - half\n\n\ndef build_compact_ids(cache, row):\n    """\n    Lazy V12 equivalent of the old precomputed lengths/cat_start/attr_start.\n\n    It touches only this one 256-token row and preserves the old exact rule:\n    if both labels are found, remove the Category block and keep\n    [BOS + Name ...] + [Attributes ... + EOS].\n    """\n    ids_row_full = np.asarray(cache["input_ids"][row])\n    attn_row = np.asarray(cache["attention_mask"][row])\n    item_length = int(attn_row.sum())\n\n    if item_length <= 0:\n        return np.empty(0, dtype=np.int32)\n\n    ids_row = ids_row_full[:item_length]\n\n    if item_length >= 2:\n        cat_hits = np.flatnonzero(\n            (ids_row[:-1] == CATEGORY_LABEL_TOKEN)\n            & (ids_row[1:] == LABEL_COLON_TOKEN)\n        )\n        attr_hits = np.flatnonzero(\n            (ids_row[:-1] == ATTRIBUTES_LABEL_TOKEN)\n            & (ids_row[1:] == LABEL_COLON_TOKEN)\n        )\n        category_start = int(cat_hits[0]) if len(cat_hits) else -1\n        attr_start = int(attr_hits[0]) if len(attr_hits) else -1\n    else:\n        category_start = -1\n        attr_start = -1\n\n    if category_start < 0 or attr_start < 0:\n        return ids_row\n\n    return np.concatenate(\n        (ids_row[:category_start], ids_row[attr_start:item_length])\n    )\n\n\nclass PairIndexDataset(Dataset):\n    def __init__(self, scope, row1, row2, labels, categories):\n        self.scope = scope\n        self.row1 = row1\n        self.row2 = row2\n        self.labels = labels\n        self.categories = categories\n\n    def __len__(self):\n        return len(self.row1)\n\n    def __getitem__(self, i):\n        return self.scope[i], self.row1[i], self.row2[i], self.labels[i], self.categories[i]\n\n\ndef make_pair_collate_fn(caches, pad_id, pair_max_length):\n    def collate_fn(batch):\n        scope, row1, row2, labels, categories = zip(*batch)\n        bsz = len(batch)\n        input_ids = np.full((bsz, pair_max_length), pad_id, dtype=np.int64)\n        attention_mask = np.zeros((bsz, pair_max_length), dtype=np.int64)\n        for i in range(bsz):\n            cache = caches[int(scope[i])]\n            c1 = build_compact_ids(cache, int(row1[i]))\n            c2 = build_compact_ids(cache, int(row2[i]))\n            l1, l2 = len(c1), len(c2)\n            if l1 + l2 <= pair_max_length:\n                k1, k2 = l1, l2\n            else:\n                k1, k2 = allocate_pair_budget(l1, l2, pair_max_length)\n            combined = np.concatenate((c1[:k1], c2[:k2]))\n            n = len(combined)\n            input_ids[i, :n] = combined\n            attention_mask[i, :n] = 1\n        # scope rides along (0=human, 1=llm) so the training loop can apply\n        # HUMAN_WEIGHT without a second lookup.\n        return (torch.from_numpy(input_ids), torch.from_numpy(attention_mask),\n                torch.tensor(labels, dtype=torch.float32),\n                np.asarray(categories, dtype=np.int64),\n                torch.tensor([int(s) for s in scope], dtype=torch.int64))\n    return collate_fn\n\n\ndef make_train_loader(frame, caches, pad_id, per_core_batch, num_replicas, rank, seed, epoch):\n    dataset = PairIndexDataset(frame["scope"], frame["row1"], frame["row2"],\n                               frame["label"], frame["category"])\n    sampler = DistributedSampler(dataset, num_replicas=num_replicas, rank=rank, shuffle=True,\n                                 seed=seed, drop_last=True)\n    sampler.set_epoch(epoch)\n    loader = DataLoader(dataset, batch_size=per_core_batch, sampler=sampler, drop_last=True,\n                        collate_fn=make_pair_collate_fn(caches, pad_id, PAIR_MAX_LENGTH),\n                        num_workers=0)\n    return loader, sampler\n\n\ndef make_eval_loader(frame, caches, pad_id, eval_batch):\n    dataset = PairIndexDataset(frame["scope"], frame["row1"], frame["row2"],\n                               frame["label"], frame["category"])\n    collate_fn = make_pair_collate_fn(caches, pad_id, PAIR_MAX_LENGTH)\n    n = len(dataset)\n    n_full = (n // eval_batch) * eval_batch\n    full_loader = DataLoader(torch.utils.data.Subset(dataset, list(range(n_full))),\n                             batch_size=eval_batch, shuffle=False, drop_last=True,\n                             collate_fn=collate_fn, num_workers=0)\n    remainder_loader = None\n    if n_full < n:\n        remainder_loader = DataLoader(torch.utils.data.Subset(dataset, list(range(n_full, n))),\n                                      batch_size=n - n_full, shuffle=False, drop_last=False,\n                                      collate_fn=collate_fn, num_workers=0)\n    return full_loader, remainder_loader\n\n\n# =====================================================================\n# MODEL\n# =====================================================================\n\nclass LinearHead(nn.Module):\n    """One Linear per category. The merged backbone already carries the\n    general model\'s capacity, so a specialist head only has to place its own\n    category\'s decision boundary -- and with ~115K rows per category, a\n    deeper head would mostly memorize."""\n\n    def __init__(self, hidden_dim, dropout=HEAD_DROPOUT):\n        super().__init__()\n        self.dropout = nn.Dropout(dropout)\n        self.out = nn.Linear(hidden_dim, 1)\n\n    def forward(self, x):\n        return self.out(self.dropout(x)).view(-1)\n\n\nclass FridaCrossEncoderClassifier(nn.Module):\n    def __init__(self, encoder, head):\n        super().__init__()\n        self.encoder = encoder\n        self.classifier = head\n\n    def forward(self, input_ids, attention_mask):\n        with torch.autocast("xla", dtype=torch.bfloat16):\n            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)\n        cls = out.last_hidden_state[:, 0, :].float()\n        return self.classifier(cls).view(-1).float()\n\n\n# The already-published dataset holding a verified merge + 3 trained\n# specialists (categories 07/10/14 -- confirmed by directly listing the\n# dataset\'s files, NOT by trusting the "8, 10, 14" recalled verbally;\n# the actual successful categories are 07/10/14). Its own internal\n# layout is frida_percat_lora/{merged_base, cat{NN}_best, cat{NN}_last}/,\n# i.e. a direct publish of an earlier run\'s own OUTPUT_DIR.\nEXTERNAL_CHECKPOINTS_HINT = (\n    "/kaggle/input/datasets/maxamsecret/"\n    "frida-with-merged-lora-and-8-10-14-cat-loras/frida_percat_lora"\n)\n\n\ndef _find_external_checkpoints_root():\n    if os.path.isdir(EXTERNAL_CHECKPOINTS_HINT):\n        return EXTERNAL_CHECKPOINTS_HINT\n    if os.path.isdir("/kaggle/input"):\n        for root, dirs, _files in os.walk("/kaggle/input"):\n            if "merged_base" in dirs and any(d.endswith("_best") for d in dirs):\n                return root\n    return None\n\n\ndef _find_external_merged_base_dir():\n    root = _find_external_checkpoints_root()\n    if root is None:\n        return None\n    candidate = os.path.join(root, "merged_base")\n    return candidate if os.path.isfile(os.path.join(candidate, "config.json")) else None\n\n\ndef _find_external_category_checkpoint(cat_id):\n    """Returns the external dataset\'s cat{cat_id:02d}_best/ dir IF it has a\n    real (non-null) val_pr_auc in its own train_state.json, else None --\n    the same explicit-null-means-unvalidated convention save_checkpoint\n    itself uses, so a checkpoint that merely exists but never finished\n    validating (or was itself a NaN casualty saved under _last) can never\n    be mistaken for a good one. Used by v5-style \'reuse, don\'t retrain\'\n    logic and by v6\'s resume-preference (best over last)."""\n    root = _find_external_checkpoints_root()\n    if root is None:\n        return None\n    cat_dir = os.path.join(root, f"cat{cat_id:02d}_best")\n    state_path = os.path.join(cat_dir, "train_state.json")\n    if not os.path.isfile(state_path):\n        return None\n    with open(state_path) as f:\n        state = json.load(f)\n    if state.get("val_pr_auc") is None:\n        return None\n    return cat_dir, state["val_pr_auc"]\n\n\ndef _find_any_external_category_checkpoint(cat_id):\n    """v6: resume source for EVERY category, not just the validated ones.\n    Prefers cat{NN}_best (real val_pr_auc); falls back to cat{NN}_last\n    (a quiet mid-training save, val_pr_auc=null) if that\'s the only thing\n    v5\'s own session produced for this category before its wall-clock cutoff\n    -- any already-trained state is a better starting point than a fresh\n    random init, per direct instruction (\'Инициализация существующими\n    адаптерами+головами\'). Returns (checkpoint_dir, source_label,\n    val_pr_auc_or_None) or None if nothing exists for this category at all."""\n    best = _find_external_category_checkpoint(cat_id)\n    if best is not None:\n        return best[0], "best", best[1]\n    root = _find_external_checkpoints_root()\n    if root is None:\n        return None\n    cat_dir = os.path.join(root, f"cat{cat_id:02d}_last")\n    if not os.path.isdir(cat_dir):\n        return None\n    return cat_dir, "last", None\n\n\ndef merge_and_save_base(resume_dir):\n    """Merges the validated general adapter into the base weights and writes\n    the result to MERGED_BASE_DIR *before any training happens*.\n\n    Runs pre-fork on CPU: 8 replicas each re-merging the same thing would be\n    pure waste, and /kaggle/working is persisted by Kaggle even when a\n    session is cancelled or hard-capped, so a crash three hours into\n    training still leaves these weights recoverable.\n\n    Idempotent -- a re-run that finds a finished merge reuses it. Also\n    reuses the merge already published in the EXTERNAL 8/10/14-checkpoint\n    dataset when present, per direct instruction (\'Mergенные веса Frida...\n    будут использоваться\') -- the merge is deterministic given the same\n    adapter, but copying ~15s of already-done work is still cheaper and\n    removes any chance of a subtly different re-merge."""\n    marker = os.path.join(MERGED_BASE_DIR, "config.json")\n    if os.path.isfile(marker):\n        print(f"[{_elapsed():8.1f}s][MERGE] reusing existing merge at {MERGED_BASE_DIR}", flush=True)\n        return MERGED_BASE_DIR\n\n    external_merge_dir = _find_external_merged_base_dir()\n    if external_merge_dir is not None:\n        print(f"[{_elapsed():8.1f}s][MERGE] reusing already-published merge from "\n              f"{external_merge_dir} (not re-merging)", flush=True)\n        shutil.copytree(external_merge_dir, MERGED_BASE_DIR)\n        return MERGED_BASE_DIR\n\n    print(f"[{_elapsed():8.1f}s][MERGE] merging {resume_dir} into base weights ...", flush=True)\n    try:\n        base = T5EncoderModel.from_pretrained(MODEL_NAME, dtype=torch.bfloat16,\n                                               attn_implementation="eager")\n    except (TypeError, ValueError):\n        base = T5EncoderModel.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)\n\n    # Snapshot one real weight so the merge can be PROVEN to have changed the\n    # model. A silent no-op merge (wrong dir, adapter that never loaded) would\n    # otherwise train 20 specialists on top of the plain base model and look\n    # completely normal in the logs -- exactly the class of silent failure\n    # this project keeps paying for.\n    probe_name = "encoder.block.0.layer.0.SelfAttention.q.weight"\n    before = dict(base.named_parameters())[probe_name].detach().clone().float()\n\n    peft_model = PeftModel.from_pretrained(base, resume_dir, is_trainable=False)\n    merged = peft_model.merge_and_unload()\n\n    after = dict(merged.named_parameters())[probe_name].detach().float()\n    delta = (after - before).abs().max().item()\n    if delta == 0.0:\n        raise RuntimeError(\n            f"[MERGE] merge was a NO-OP: {probe_name} is byte-identical before and "\n            f"after merge_and_unload(). The adapter at {resume_dir} did not apply.")\n    print(f"[{_elapsed():8.1f}s][MERGE] verified: {probe_name} max|delta|={delta:.6f}", flush=True)\n\n    os.makedirs(MERGED_BASE_DIR, exist_ok=True)\n    merged.save_pretrained(MERGED_BASE_DIR, safe_serialization=True)\n    with open(os.path.join(MERGED_BASE_DIR, "MERGE_PROVENANCE.json"), "w") as f:\n        json.dump({"base_model": MODEL_NAME, "merged_adapter_dir": resume_dir,\n                   "probe_param": probe_name, "probe_max_abs_delta": delta,\n                   "dtype": "bfloat16"}, f, indent=2)\n    del peft_model, merged, base\n    gc.collect()\n    print(f"[{_elapsed():8.1f}s][MERGE] saved -> {MERGED_BASE_DIR}", flush=True)\n    return MERGED_BASE_DIR\n\n\ndef build_specialist_model_cpu(merged_dir):\n    """Builds the merged-base + fresh-LoRA + head model ONCE, on CPU, BEFORE\n    fork -- meant to be wrapped in xmp.MpModelWrapper by the caller so all 8\n    ranks share this ONE set of host-memory pages via fork\'s copy-on-write,\n    instead of each rank independently calling T5EncoderModel.from_pretrained\n    (a real disk read + fresh CPU allocation) the way build_specialist_model\n    used to.\n\n    REAL CRASH (host RAM OOM, three real runs, 2026-08-22): PyTorch/XLA\'s own\n    docs describe this exact anti-pattern -- "instead of creating models on\n    each multiprocessing process ..., the model is created once at global\n    scope, and then moved into each device inside the spawn() target\n    function" -- and this kernel was doing precisely the thing being warned\n    against: model construction lived inside _mp_fn (per-rank, post-fork),\n    while _load_shared_data()\'s own DATA loading already correctly ran\n    pre-fork. Splitting this out is the fix: build here (pre-fork, wrapped),\n    only `.to(device)` happens per-rank in _mp_fn now -- a cheap transfer of\n    already-loaded pages, not a fresh load. Untested against the OOM\n    directly (this is one of two independent fixes being tried in parallel,\n    per direct instruction, specifically so the effect of each is\n    separable) -- see frida_percat_lora\'s sibling push (maximmarkovbb,\n    init_lora_weights="olora" + smaller batch) for the other."""\n    try:\n        base = T5EncoderModel.from_pretrained(merged_dir, dtype=torch.bfloat16,\n                                               attn_implementation="eager")\n    except (TypeError, ValueError):\n        base = T5EncoderModel.from_pretrained(merged_dir, dtype=torch.bfloat16)\n    hidden_dim = base.config.d_model\n\n    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,\n                          target_modules=LORA_TARGET_MODULES, bias="none",\n                          task_type=TaskType.FEATURE_EXTRACTION)\n    peft_model = get_peft_model(base, lora_cfg)\n    for name, p in peft_model.named_parameters():\n        if "lora_" in name:\n            p.data = p.data.float()\n            p.requires_grad = True\n    n_trainable = sum(p.numel() for n, p in peft_model.named_parameters()\n                      if "lora_" in n and p.requires_grad)\n    assert n_trainable > 0, "fresh LoRA has zero trainable params"\n\n    model = FridaCrossEncoderClassifier(peft_model, LinearHead(hidden_dim))\n    return model, hidden_dim, n_trainable\n\n\nclass TrainingDivergedError(RuntimeError):\n    """Raised the moment a step\'s loss is non-finite. Deliberately a\n    DIFFERENT exception type than the generic ones the step-level\n    try/except already tolerates (a genuinely transient/recoverable\n    single-step glitch) -- see the training loop for how this is let\n    through instead of being silently caught and skipped like an ordinary\n    failed step. Per direct instruction: a NaN loss must fail loudly with\n    a clear error, not silently keep "training" on garbage for the rest\n    of that category\'s ~600 steps only to discover the problem at eval\n    time (average_precision_score raising on NaN predictions) -- which is\n    exactly what the previous version did, and it also wasted real TPU\n    wall-clock: every one of the 17 failing categories in the real run\n    burned its full step budget before failing."""\n\n\ndef reset_specialist_(model, seed):\n    """Returns the adapter + head to a fresh initialisation, in place.\n\n    peft\'s own convention is lora_A ~ kaiming_uniform, lora_B = 0, so the\n    adapter starts as an exact identity and the first steps see the merged\n    model unchanged -- which is what makes starting from a strong backbone\n    safe. Reproduced here rather than rebuilding the module (see\n    build_specialist_model_cpu on why the module is reused)."""\n    torch.manual_seed(seed)\n    for name, p in model.encoder.named_parameters():\n        if "lora_A" in name:\n            nn.init.kaiming_uniform_(p, a=math.sqrt(5))\n        elif "lora_B" in name:\n            nn.init.zeros_(p)\n    for m in model.classifier.modules():\n        if isinstance(m, nn.Linear):\n            nn.init.normal_(m.weight, std=0.02)\n            nn.init.zeros_(m.bias)\n\n\ndef load_specialist_from_checkpoint_(model, checkpoint_dir):\n    """Loads an existing category\'s adapter + head weights into the\n    (already-built, reused-across-categories) model IN PLACE -- v6\'s\n    replacement for reset_specialist_ when a real prior checkpoint exists\n    for this category (see _find_any_external_category_checkpoint).\n\n    Verifies the load actually changed something, same reasoning as\n    merge_and_save_base\'s own probe-delta check: a silent no-op load\n    (wrong dir, key mismatch swallowed by strict=False) would otherwise\n    train on top of the FRESH init from build_specialist_model_cpu\'s own\n    get_peft_model() call and look completely normal in the logs."""\n    adapter_dir = os.path.join(checkpoint_dir, "adapter")\n    head_path = os.path.join(checkpoint_dir, "head.pt")\n    if not os.path.isfile(os.path.join(adapter_dir, "adapter_model.safetensors")):\n        raise RuntimeError(f"[RESUME] no adapter_model.safetensors under {adapter_dir}")\n    if not os.path.isfile(head_path):\n        raise RuntimeError(f"[RESUME] no head.pt at {head_path}")\n\n    probe_name = next(n for n, p in model.encoder.named_parameters() if "lora_A" in n)\n    before = dict(model.encoder.named_parameters())[probe_name].detach().clone()\n\n    adapter_sd = safetensors_load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))\n    load_result = set_peft_model_state_dict(model.encoder, adapter_sd)\n    missing = getattr(load_result, "missing_keys", None) or []\n    unexpected = getattr(load_result, "unexpected_keys", None)\n    # set_peft_model_state_dict does model.load_state_dict(adapter_sd, strict=False)\n    # under the hood, and adapter_sd only ever contains the LoRA delta weights (lora_A/\n    # lora_B/...) -- every base-model weight (base_model.model.*, *.base_layer.weight,\n    # relative_attention_bias, layer_norm, ...) is CORRECTLY reported "missing" by\n    # strict=False every single time, real crash confirmed against a live run: every\n    # one of ~230 "missing" keys was a base weight, zero were lora_ keys, the adapter\n    # itself loaded fine. Only a missing LORA key is a real problem (rank/target-module\n    # mismatch); missing base keys are how a partial adapter load is SUPPOSED to look.\n    missing_lora = [k for k in missing if "lora_" in k]\n    if missing_lora:\n        raise RuntimeError(f"[RESUME] adapter load from {adapter_dir} missing LoRA keys: "\n                           f"{missing_lora}")\n    if unexpected:\n        raise RuntimeError(f"[RESUME] adapter load from {adapter_dir} unexpected keys: {unexpected}")\n\n    head_sd = torch.load(head_path, map_location="cpu")\n    model.classifier.load_state_dict(head_sd, strict=True)\n\n    # Real crash: cat00_last\'s own weights were already non-finite (this\n    # dataset\'s non-_best categories are the ORIGINAL pre-NaN-fix run\'s\n    # casualties -- see the NaN root-cause section above; only 07/10/14 ever\n    # reached a validated _best there). Loading them produced loss=nan at\n    # step 1 of the very next training step, which used to crash the WHOLE\n    # multi-category run (see the REAL CRASH note above on\n    # TrainingDivergedError never actually being caught at category level).\n    # Catch it HERE instead, before a single training step runs on it, so\n    # the caller can fall back to a fresh init instead of training on\n    # garbage.\n    bad = [n for n, p in model.encoder.named_parameters()\n           if "lora_" in n and not torch.isfinite(p).all()]\n    bad += [n for n, p in model.classifier.named_parameters()\n            if not torch.isfinite(p).all()]\n    if bad:\n        raise RuntimeError(f"[RESUME] checkpoint at {checkpoint_dir} has non-finite "\n                           f"values in {len(bad)} param(s) (e.g. {bad[0]}) -- refusing "\n                           f"to resume from a corrupted checkpoint.")\n\n    after = dict(model.encoder.named_parameters())[probe_name].detach()\n    delta = (after.float() - before.float()).abs().max().item()\n    if delta == 0.0:\n        raise RuntimeError(\n            f"[RESUME] load was a NO-OP: {probe_name} is byte-identical before and "\n            f"after loading from {checkpoint_dir} -- the checkpoint did not apply.")\n    for name, p in model.encoder.named_parameters():\n        if "lora_" in name:\n            p.data = p.data.float()\n            p.requires_grad = True\n    return delta\n\n\ndef make_param_groups(model, lr_scale=1.0):\n    """Two groups so the randomly-initialised head and the warm backbone can\n    follow different schedules -- see the docstring\'s HEAD SCHEDULE section."""\n    lora_params = [p for n, p in model.encoder.named_parameters()\n                   if "lora_" in n and p.requires_grad]\n    head_params = [p for p in model.classifier.parameters() if p.requires_grad]\n    return [{"params": lora_params, "lr": LR * lr_scale, "name": "lora"},\n            {"params": head_params, "lr": LR * lr_scale * HEAD_LR_MULTIPLIER, "name": "head"}]\n\n\ndef _make_head_lr_lambda(warmup_steps, total_steps, min_ratio):\n    """Head ramps to its (higher) peak over a SHORT warmup\n    (HEAD_WARMUP_STEPS, much shorter than the adapter\'s own warmup_steps),\n    then holds that peak flat until the adapter\'s own warmup ends, then\n    joins the same cosine decay.\n\n    NOT an instant jump to peak at step 0 -- that was the shipped version\n    and produced loss=nan from the very first logged step on most\n    categories, reproducibly (see HEAD_LR_MULTIPLIER\'s own comment for the\n    real evidence). A step-0 jump to a multiplied LR on a freshly\n    -initialised linear head has no gradient history to scale against; a\n    short ramp (still much faster than the adapter\'s) plus gradient\n    clipping in the training loop fixes it."""\n    head_warmup = max(1, min(HEAD_WARMUP_STEPS, warmup_steps))\n\n    def lr_lambda(step):\n        if step < head_warmup:\n            return step / head_warmup\n        if step < warmup_steps:\n            return 1.0\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        progress = min(progress, 1.0)\n        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))\n    return lr_lambda\n\n\ndef _make_lr_lambda(warmup_steps, total_steps, min_ratio):\n    """Linear warmup then cosine decay to a FLOOR of min_ratio * peak, never\n    to zero. Written out with LambdaLR (core PyTorch) rather than\n    transformers\' cosine_with_min_lr, whose availability depends on the\n    transformers version the image happens to ship -- and TPU-image version\n    drift has already killed one kernel in this project."""\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return step / max(1, warmup_steps)\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\n        progress = min(progress, 1.0)\n        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))\n    return lr_lambda\n\n\n# =====================================================================\n# METRICS + EVAL\n# =====================================================================\n\ndef compute_metrics(probs, labels, categories, category_names):\n    per_cat = {}\n    for cat_id, cat_name in enumerate(category_names):\n        mask = categories == cat_id\n        if mask.sum() < 2:\n            continue\n        y_true = labels[mask]\n        if len(np.unique(y_true)) < 2:\n            continue\n        per_cat[cat_name] = average_precision_score(y_true, probs[mask])\n    macro = float(np.mean(list(per_cat.values()))) if per_cat else float("nan")\n    return macro, per_cat\n\n\ndef evaluate(model, full_loader, remainder_loader, device, category_names):\n    model.eval()\n    all_logits, all_labels, all_categories = [], [], []\n    with torch.no_grad():\n        for loader in (full_loader, remainder_loader):\n            if loader is None:\n                continue\n            for input_ids, attention_mask, batch_labels, batch_categories, _scope in loader:\n                logits = model(input_ids=input_ids.to(device),\n                               attention_mask=attention_mask.to(device))\n                all_logits.append(logits.cpu())\n                all_labels.append(batch_labels)\n                all_categories.append(batch_categories)\n    logits = torch.cat(all_logits, dim=0)\n    labels = torch.cat(all_labels, dim=0)\n    categories = np.concatenate(all_categories, axis=0)\n    macro, per_cat = compute_metrics(torch.sigmoid(logits).numpy(), labels.numpy(),\n                                     categories, category_names)\n    model.train()\n    return {"macro_pr_auc": macro, "per_category": per_cat, "n": len(labels)}\n\n\n# =====================================================================\n# CHECKPOINT -- own directory per save, explicit null when unvalidated\n# =====================================================================\n\ndef save_checkpoint(model, optimizer, global_step, epoch, val_macro_pr_auc, label, rank,\n                     extra=None):\n    if rank != 0:\n        return\n    save_dir = os.path.join(OUTPUT_DIR, label)\n    xm.mark_step()\n    os.makedirs(save_dir, exist_ok=True)\n\n    adapter_dir = os.path.join(save_dir, "adapter")\n    os.makedirs(adapter_dir, exist_ok=True)\n    lora_sd = get_peft_model_state_dict(model.encoder)\n    safetensors_save_file({k: v.detach().cpu() for k, v in lora_sd.items()},\n                          os.path.join(adapter_dir, "adapter_model.safetensors"))\n    model.encoder.peft_config["default"].save_pretrained(adapter_dir)\n\n    torch.save({k: v.detach().cpu() for k, v in model.classifier.state_dict().items()},\n               os.path.join(save_dir, "head.pt"))\n\n    # V12 host-RAM safety: resume logic in this file never reloads optimizer.pt;\n    # it only reloads adapter + head and intentionally creates a fresh optimizer.\n    # Serializing XLA Adam state therefore bought nothing while creating a large\n    # host-side transient allocation exactly where RAM is already the failure mode.\n    optimizer_saved = False\n\n    with open(os.path.join(save_dir, "train_state.json"), "w") as f:\n        json.dump({\n            "epoch": epoch, "global_step": global_step,\n            "val_pr_auc": val_macro_pr_auc,   # explicit null if unvalidated\n            "category": extra.get("category") if extra else None,\n            "head": {"type": "LinearHead", "dropout": HEAD_DROPOUT},\n            "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,\n                     "target_modules": LORA_TARGET_MODULES},\n            "pair_max_length": PAIR_MAX_LENGTH,\n            "merged_base": "merged_base/",\n            "optimizer_saved": optimizer_saved,\n            **(extra or {}),\n        }, f, indent=2)\n    print(f"[SAVE] {label}: epoch={epoch} step={global_step} "\n          f"val_macro_pr_auc={val_macro_pr_auc}", flush=True)\n\n\n# =====================================================================\n# SHARED CPU-ONLY SETUP -- ONCE, pre-fork\n# =====================================================================\n\n_SHARED = {}\n# Set once in main(), pre-fork, via xmp.MpModelWrapper -- see\n# build_specialist_model_cpu\'s own docstring for why. Global (not folded\n# into _SHARED) to match torch_xla\'s own documented MpModelWrapper pattern\n# exactly, so a future reader who\'s seen that pattern recognizes it here.\n_WRAPPED_MODEL = None\n\n\ndef _build_frame(id1, id2, label, category, scope, caches):\n    cache = caches[scope]\n    row1 = lookup_rows(cache, np.asarray(id1, dtype=np.int64))\n    row2 = lookup_rows(cache, np.asarray(id2, dtype=np.int64))\n    missing = int((row1 < 0).sum() + (row2 < 0).sum())\n    if missing:\n        raise RuntimeError(f"[SETUP] {missing:,} endpoints missing from the "\n                           f"{cache[\'name\']} item cache -- wrong cache scope?")\n    return dict(scope=np.full(len(row1), scope, dtype=np.int8), row1=row1, row2=row2,\n                label=np.asarray(label, dtype=np.float32),\n                category=np.asarray(category, dtype=np.int16))\n\n\ndef _concat_frames(a, b):\n    return {k: np.concatenate([a[k], b[k]]) for k in a}\n\n\ndef _index_frame(frame, mask):\n    return {k: v[mask] for k, v in frame.items()}\n\n\ndef stratified_llm_sample(target, cat_id, n_per_category, seed):\n    """Picks up to n_per_category LLM rows for EVERY category, preserving that\n    category\'s own target-value distribution exactly.\n\n    Why this and not a confidence filter (the alternative that was\n    considered and rejected): matches_llm targets are not continuous, they\n    are 10 discrete levels k/9 -- a 9-judge vote count. Measured over all\n    11,187,780 rows:\n\n        0.000  64.22%   |  0.556   2.41%\n        0.111   0.01%   |  0.667   1.52%\n        0.222   5.65%   |  0.778   2.47%\n        0.333   4.44%   |  0.889   6.08%\n        0.444   2.26%   |  1.000  10.94%\n\n    So "confident" rows (<=0.2 or >=0.8) are ALREADY 81.25% of the data by\n    nature. Filtering on confidence would push that to 100% while discarding\n    the 18.75% of rows sitting nearest the decision boundary -- and PR-AUC is\n    a ranking metric, decided precisely there. It would also be the worst\n    possible moment to do it: the backbone merged into this run is\n    frida-tpu-human-sft, trained human-ONLY, so these specialists are the\n    model\'s first exposure to LLM-labelled data at all.\n\n    Category balance is enforced (every category gets the same quota) rather\n    than sampled proportionally, because the metric is a plain mean over\n    categories -- each one is equally worth getting right.\n    """\n    rng = np.random.RandomState(seed)\n    keep = []\n    realized = {}\n    for c in np.unique(cat_id):\n        cat_rows = np.flatnonzero(cat_id == c)\n        n_take = min(n_per_category, len(cat_rows))\n        cat_targets = target[cat_rows]\n        levels, level_counts = np.unique(cat_targets, return_counts=True)\n        # proportional allocation, then fix the rounding drift on the largest\n        # level so the total lands exactly on n_take\n        quota = np.floor(level_counts / level_counts.sum() * n_take).astype(np.int64)\n        deficit = n_take - int(quota.sum())\n        if deficit > 0:\n            quota[np.argsort(-level_counts)[:deficit]] += 1\n        picked = []\n        for level, q in zip(levels, quota):\n            if q <= 0:\n                continue\n            pool = cat_rows[cat_targets == level]\n            q = min(q, len(pool))\n            picked.append(rng.choice(pool, size=q, replace=False))\n        sel = np.concatenate(picked) if picked else np.array([], dtype=np.int64)\n        keep.append(sel)\n        realized[int(c)] = {"taken": int(len(sel)), "available": int(len(cat_rows)),\n                            "levels": {float(l): int(q) for l, q in zip(levels, quota) if q > 0}}\n    return np.concatenate(keep), realized\n\n\ndef _load_shared_data():\n    print(f"[{_elapsed():8.1f}s][SETUP] loading all CPU-only data once, pre-fork ...", flush=True)\n\n    # ---- fail cheap: everything the run cannot proceed without, first ----\n    resume_dir = _find_resume_adapter_dir()\n    if resume_dir is None:\n        raise RuntimeError("[SETUP] no adapter_config.json under /kaggle/input -- mount the "\n                           "validated general FRIDA adapter via dataset_sources.")\n    os.makedirs(OUTPUT_DIR, exist_ok=True)\n\n    # ---- the merge, saved BEFORE any training (see docstring) ----\n    merged_dir = merge_and_save_base(resume_dir)\n\n    human_cache = load_v1_cache(HUMAN_CACHE_HINT, "human")\n    llm_cache = load_v1_cache(LLM_CACHE_HINT, "llm")\n    if human_cache["pad_id"] != llm_cache["pad_id"]:\n        raise RuntimeError(f"[SETUP] pad_id differs between caches: "\n                           f"{human_cache[\'pad_id\']} vs {llm_cache[\'pad_id\']}")\n    caches = [human_cache, llm_cache]\n    pad_id = human_cache["pad_id"]\n\n    # ---- human: the OLD canonical 80/20 split, deliberately (see docstring) ----\n    matches_pd = (\n        pl.read_parquet(_find_dataset_file(MATCHES_PATH_HINT, "matches.parquet"))\n        .filter(pl.col("id1").is_not_null() & pl.col("id2").is_not_null())\n        .with_columns([pl.col("target").fill_null(0).cast(pl.Float32),\n                       (pl.col("target") >= 0.5).cast(pl.Int8).alias("label")])\n        .select(["id1", "id2", "label"])\n        .to_pandas()\n    )\n    labels_all = matches_pd["label"].to_numpy().astype(np.int64)\n    indices = np.arange(len(labels_all), dtype=np.int64)\n    train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=SEED,\n                                          stratify=labels_all)\n    human_train = matches_pd.iloc[train_idx].reset_index(drop=True)\n    human_val = matches_pd.iloc[val_idx].reset_index(drop=True)\n    val_fingerprint = hashlib.sha256(\n        ",".join(f"{a}-{b}" for a, b in sorted(zip(human_val["id1"], human_val["id2"]))).encode("utf-8")\n    ).hexdigest()[:16]\n    print(f"[{_elapsed():8.1f}s][SETUP] human train={len(human_train):,} val={len(human_val):,} "\n          f"val_fingerprint={val_fingerprint} (OLD canonical 80/20 split -- see docstring)",\n          flush=True)\n\n    items_cat = (\n        pl.scan_parquet(_find_dataset_file(ITEMS_HUMAN_PATH_HINT, "items_human.parquet"))\n        .select(["id", "category"])\n        .with_columns(pl.col("category").fill_null("UNKNOWN").cast(pl.Utf8)\n                      .str.slice(0, MAX_CATEGORY_CHARS))\n        .collect(engine="streaming")\n    )\n    category_vocab = sorted(items_cat.get_column("category").unique().to_list())\n    category_to_id = {c: i for i, c in enumerate(category_vocab)}\n    id_to_category = dict(zip(items_cat.get_column("id").to_list(),\n                              items_cat.get_column("category").to_list()))\n    print(f"[{_elapsed():8.1f}s][SETUP] categories: {len(category_vocab)}", flush=True)\n\n    def cat_ids(frame):\n        # No UNKNOWN fallback on purpose. items_human.parquet is built to cover\n        # every id matches.parquet references, and a local check over all\n        # 365,654 pairs found both endpoints sharing a real category in 100.0%\n        # of them. If that ever stops holding, an unmapped id silently becoming\n        # "UNKNOWN" would quietly move rows into a 21st pseudo-category and\n        # corrupt the per-category split -- so fail loudly instead.\n        mapped = frame["id1"].map(id_to_category).map(category_to_id)\n        n_missing = int(mapped.isna().sum())\n        if n_missing:\n            raise RuntimeError(f"[SETUP] {n_missing:,} pairs have an id1 with no category in "\n                               f"items_human.parquet -- the per-category split cannot be built.")\n        return mapped.to_numpy().astype(np.int64)\n\n    human_train_frame = _build_frame(human_train["id1"], human_train["id2"],\n                                     human_train["label"], cat_ids(human_train), 0, caches)\n    human_val_frame = _build_frame(human_val["id1"], human_val["id2"],\n                                   human_val["label"], cat_ids(human_val), 0, caches)\n    del matches_pd, items_cat, id_to_category, labels_all, indices, train_idx, val_idx\n    gc.collect()\n\n    # ---- LLM: full train split of frida-llm-train-val-split-no-leak-\n    # retroactive (v2), not a fixed subsample -- per direct instruction\n    # ("Обучение не на llm-subsample, а на всей train-части llm-разметки").\n    # That kernel\'s own "train" already means "every row not held out for\n    # its fixed 100K val" (~11.09M rows); category balance/stratification\n    # was already applied there (category -> positive_rate-within-category\n    # -> length-within-category), not re-applied here. Its "category"\n    # column is the string name (same category_vocab construction, per its\n    # own docstring), so it maps onto this run\'s own category_to_id\n    # directly -- no re-join against items.parquet needed at all.\n    llm_split_path = _find_dataset_file(\n        LLM_SPLIT_PATH_HINT, "llm_train_val_split.parquet"\n    )\n\n    # V12: required columns only, streaming Polars. Avoid a giant pandas frame\n    # with 11M Python category strings before the 8-way fork.\n    llm_scan_all = pl.scan_parquet(llm_split_path)\n    n_llm_total = int(\n        llm_scan_all.select(pl.len().alias("n"))\n        .collect(engine="streaming")\n        .get_column("n")[0]\n    )\n\n    llm_pl = (\n        llm_scan_all\n        .filter(pl.col("split") == "train")\n        .select(["id1", "id2", "target", "category"])\n        .with_columns(\n            pl.col("category")\n            .replace_strict(category_to_id, default=None)\n            .cast(pl.Int16)\n            .alias("cat_id")\n        )\n        .collect(engine="streaming")\n    )\n\n    n_unmapped = int(llm_pl.get_column("cat_id").null_count())\n    if n_unmapped:\n        raise RuntimeError(\n            f"[SETUP] {n_unmapped:,} llm-train rows carry a category not in "\n            f"this run\'s own category_vocab -- split/vocab mismatch, do not trust this run."\n        )\n\n    n_llm_used = len(llm_pl)\n    print(\n        f"[{_elapsed():8.1f}s][SETUP] LLM TRAIN (full split, no subsample): "\n        f"{n_llm_used:,} rows (of {n_llm_total:,} total split rows, train+val) -- "\n        f"Polars streaming, required columns only",\n        flush=True,\n    )\n\n    llm_frame = _build_frame(\n        llm_pl.get_column("id1").to_numpy(),\n        llm_pl.get_column("id2").to_numpy(),\n        llm_pl.get_column("target").to_numpy(),\n        llm_pl.get_column("cat_id").to_numpy().astype(np.int16, copy=False),\n        1,\n        caches,\n    )\n    del llm_pl, llm_scan_all\n    gc.collect()\n    _malloc_trim()\n\n    # ---- split everything into per-category frames ----\n    train_by_cat, val_by_cat = {}, {}\n    all_train = _concat_frames(human_train_frame, llm_frame)\n    del human_train_frame, llm_frame\n    gc.collect()\n    for cat_id in range(len(category_vocab)):\n        train_by_cat[cat_id] = _index_frame(all_train, all_train["category"] == cat_id)\n        val_by_cat[cat_id] = _index_frame(human_val_frame, human_val_frame["category"] == cat_id)\n    del all_train, human_val_frame\n    gc.collect()\n\n    for cat_id, name in enumerate(category_vocab):\n        tr, va = train_by_cat[cat_id], val_by_cat[cat_id]\n        n_human = int((tr["scope"] == 0).sum())\n        print(f"[{_elapsed():8.1f}s][SETUP]   [{cat_id:02d}] {name[:34]:<34} "\n              f"train={len(tr[\'label\']):>7,} (human {n_human:>6,} + llm {len(tr[\'label\']) - n_human:>7,})"\n              f"  val={len(va[\'label\']):>6,}", flush=True)\n\n    with open(os.path.join(OUTPUT_DIR, "llm_split_usage_report.json"), "w", encoding="utf-8") as f:\n        json.dump({\n            "policy": "v6: full train split of frida-llm-train-val-split-no-leak-"\n                       "retroactive v2, no per-category cap -- stratification "\n                       "(category / positive_rate-within-category / length-within-"\n                       "category) was already applied by that kernel, not here",\n            "llm_split_path_hint": LLM_SPLIT_PATH_HINT,\n            "llm_rows_total_in_split": int(n_llm_total),\n            "llm_rows_used_train": int(n_llm_used),\n        }, f, ensure_ascii=False, indent=2)\n\n    _SHARED.update(dict(\n        caches=caches, pad_id=pad_id, train_by_cat=train_by_cat, val_by_cat=val_by_cat,\n        category_vocab=category_vocab, val_fingerprint=val_fingerprint,\n        merged_dir=merged_dir, n_llm_total=int(n_llm_total), n_llm_used=int(n_llm_used),\n    ))\n    print(f"[{_elapsed():8.1f}s][SETUP] done -- ready for fork", flush=True)\n\n\n# =====================================================================\n# SMOKE TEST -- catch an HBM-fit / shape-compile problem in ~1-2 minutes,\n# not category N. Adapted from frida-full-sft\'s own (that kernel earned\n# this the hard way -- five real crashes, each now caught here instead of\n# hours in). frida-percat-lora never had one: every real bug found in this\n# kernel\'s history (three separate checkpoint-path bugs, the over-strict\n# missing-keys check, the DataLoader cleanup leak) was discovered only\n# after a real, paid TPU run -- this doesn\'t fix any of those, it just\n# means the NEXT shape/memory-class bug costs 1-2 minutes instead of one.\n# =====================================================================\n\ndef _make_synthetic_batch(batch_size, pair_max_length, pad_id, vocab_size, device):\n    lengths = np.random.randint(pair_max_length // 4, pair_max_length + 1, size=batch_size)\n    input_ids = np.full((batch_size, pair_max_length), pad_id, dtype=np.int64)\n    attention_mask = np.zeros((batch_size, pair_max_length), dtype=np.int64)\n    for i, L in enumerate(lengths):\n        input_ids[i, :L] = np.random.randint(3, vocab_size, size=L)  # skip special ids 0/1/2\n        attention_mask[i, :L] = 1\n    labels = np.random.randint(0, 2, size=batch_size).astype(np.float32)\n    return (torch.from_numpy(input_ids).to(device), torch.from_numpy(attention_mask).to(device),\n            torch.from_numpy(labels).to(device))\n\n\ndef _smoke_test_train_step(model, device, pad_id, vocab_size, rank):\n    _log(rank, "SMOKE", f"warm-up TRAIN step (batch={PER_CORE_TRAIN_BATCH} seq={PAIR_MAX_LENGTH}) "\n                        f"-- forcing the real forward+backward compile now, before any real data "\n                        f"or wall-clock budget is spent", force=True)\n    t0 = time.perf_counter()\n    input_ids, attention_mask, labels = _make_synthetic_batch(\n        PER_CORE_TRAIN_BATCH, PAIR_MAX_LENGTH, pad_id, vocab_size, device)\n    loss = None\n    try:\n        model.zero_grad(set_to_none=True)\n        logits = model(input_ids=input_ids, attention_mask=attention_mask)\n        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)\n        loss.backward()\n        xm.mark_step()  # force compile+execution NOW, at the real memory cost\n    except Exception:\n        _log(rank, "SMOKE", "FAILED during warm-up TRAIN step -- this shape does not fit in HBM. "\n                            "Lower PER_CORE_TRAIN_BATCH (or PAIR_MAX_LENGTH) and relaunch. Nothing "\n                            "real was touched past this point.", force=True)\n        raise\n    finally:\n        # backward() only populates .grad, no optimizer.step() was ever\n        # called (there is no optimizer yet at this point in the run --\n        # each category builds its own), so the real merged+LoRA weights\n        # are provably bit-for-bit unchanged afterward.\n        model.zero_grad(set_to_none=True)\n    _log(rank, "SMOKE", f"warm-up TRAIN step OK ({time.perf_counter() - t0:.1f}s, "\n                        f"loss={loss.item():.4f}, discarded -- model weights unchanged)", force=True)\n\n\ndef _smoke_test_eval_step(model, device, pad_id, vocab_size, rank):\n    _log(rank, "SMOKE", f"warm-up EVAL step (batch={PER_CORE_EVAL_BATCH} seq={PAIR_MAX_LENGTH})",\n         force=True)\n    t0 = time.perf_counter()\n    input_ids, attention_mask, _labels = _make_synthetic_batch(\n        PER_CORE_EVAL_BATCH, PAIR_MAX_LENGTH, pad_id, vocab_size, device)\n    model.eval()\n    try:\n        with torch.no_grad():\n            model(input_ids=input_ids, attention_mask=attention_mask)\n            xm.mark_step()\n    except Exception:\n        _log(rank, "SMOKE", "FAILED during warm-up EVAL step -- this shape does not fit in HBM. "\n                            "Lower PER_CORE_EVAL_BATCH and relaunch.", force=True)\n        raise\n    finally:\n        model.train()\n    _log(rank, "SMOKE", f"warm-up EVAL step OK ({time.perf_counter() - t0:.1f}s)", force=True)\n\n\ndef _smoke_test_real_loader_batch(model, device, caches, pad_id, train_frame, world_size, rank):\n    """Same gap frida-full-sft\'s FIFTH REAL CRASH documented: a synthetic\n    batch can succeed while the exact same model call fails on a real\n    MpDeviceLoader-sourced one. Goes through the real make_train_loader ->\n    MpDeviceLoader -> next(iter(...)) path at the real shape. Uses a\n    THROWAWAY loader (epoch=0, never reused, category 0\'s own train frame)\n    so this cannot disturb any real category\'s own sampler state."""\n    _log(rank, "SMOKE", "warm-up REAL-LOADER batch -- same make_train_loader -> MpDeviceLoader "\n                        "path real training uses, not a synthetic batch", force=True)\n    t0 = time.perf_counter()\n    throwaway_loader, _throwaway_sampler = make_train_loader(\n        train_frame, caches, pad_id, PER_CORE_TRAIN_BATCH, world_size, rank, SEED, epoch=0)\n    mp_throwaway = pl_xla.MpDeviceLoader(throwaway_loader, device)\n    loss = None\n    try:\n        input_ids, attention_mask, batch_labels, _cats, _scope = next(iter(mp_throwaway))\n        batch_labels = batch_labels.to(device)\n        model.zero_grad(set_to_none=True)\n        logits = model(input_ids=input_ids, attention_mask=attention_mask)\n        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, batch_labels)\n        loss.backward()\n        xm.mark_step()\n    except Exception:\n        _log(rank, "SMOKE", "FAILED during warm-up REAL-LOADER batch -- this is the exact "\n                            "path/shape real training uses, so this failure WOULD have "\n                            "recurred mid-run. Nothing beyond this point was touched.", force=True)\n        raise\n    finally:\n        model.zero_grad(set_to_none=True)\n        del throwaway_loader, mp_throwaway\n        # no full gc.collect() post-fork; preserve copy-on-write sharing\n    _log(rank, "SMOKE", f"warm-up REAL-LOADER batch OK ({time.perf_counter() - t0:.1f}s, "\n                        f"loss={loss.item():.4f}, discarded -- model weights unchanged)", force=True)\n\n\n# =====================================================================\n# PER-REPLICA ENTRYPOINT\n# =====================================================================\n\ndef _mp_fn(index):\n    # Parent froze long-lived pre-fork objects. Re-enable GC only for new child\n    # objects; frozen inherited objects stay out of normal cyclic collections.\n    gc.enable()\n\n    rank = xr.global_ordinal()\n    world_size = xr.world_size()\n    device = torch_xla.device()\n    set_seed(SEED + rank)\n    os.makedirs(OUTPUT_DIR, exist_ok=True)\n\n    caches = _SHARED["caches"]\n    pad_id = _SHARED["pad_id"]\n    train_by_cat, val_by_cat = _SHARED["train_by_cat"], _SHARED["val_by_cat"]\n    category_vocab = _SHARED["category_vocab"]\n\n    _log(rank, "CONFIG", f"world_size={world_size} pair_max_length={PAIR_MAX_LENGTH} "\n                         f"lora_r={LORA_R} lr={LR} head_lr={LR * HEAD_LR_MULTIPLIER} "\n                         f"warmup={WARMUP_RATIO} human_weight={HUMAN_WEIGHT} "\n                         f"lr_resume_multiplier={LR_RESUME_MULTIPLIER} "\n                         f"llm_train_full_split={_SHARED[\'n_llm_used\']:,} "\n                         f"(of {_SHARED[\'n_llm_total\']:,} split rows)", force=True)\n\n    # Built ONCE pre-fork (see build_specialist_model_cpu + _WRAPPED_MODEL in\n    # main()); this is a cheap per-rank device transfer of already-loaded\n    # host memory, not a fresh T5EncoderModel.from_pretrained() disk read --\n    # reset between categories so XLA also only ever compiles one program.\n    model = _WRAPPED_MODEL.to(device)\n    _log(\n        rank,\n        "MEM",\n        f"after model.to(device): rss_current={_current_rss_mb():.0f}MB "\n        f"rss_peak={_peak_rss_mb():.0f}MB "\n        f"mem_available={_mem_available_mb():.0f}MB",\n        force=True,\n    )\n    _log(rank, "MODEL", f"fresh LoRA r={LORA_R} alpha={LORA_ALPHA} "\n                        f"dropout={LORA_DROPOUT} trainable={_SHARED[\'n_trainable\']:,}", force=True)\n    total_batch = PER_CORE_TRAIN_BATCH * world_size\n    _log(rank, "BATCH", f"per_core_train={PER_CORE_TRAIN_BATCH} "\n                        f"per_core_eval={PER_CORE_EVAL_BATCH} "\n                        f"total_train_batch={total_batch} (fixed, no probing)", force=True)\n\n    # Hardcoded rather than introspected off model.encoder.config: FRIDA\'s\n    # real tokenizer vocab is 93,651 (RobertaTokenizer, byte-BPE -- verified\n    # directly against the loaded tokenizer, see CLAUDE.md); the synthetic\n    # batch only needs any valid in-range id, not the exact size, and\n    # PeftModel\'s `.config` delegation to the base model isn\'t something\n    # this file has relied on anywhere else.\n    vocab_size = 90_000\n    _smoke_test_train_step(model, device, pad_id, vocab_size, rank)\n    _smoke_test_eval_step(model, device, pad_id, vocab_size, rank)\n    _smoke_test_real_loader_batch(model, device, caches, pad_id, train_by_cat[0], world_size, rank)\n    # No explicit reset needed after: every smoke step only calls\n    # .backward() then zero_grad(set_to_none=True), never an optimizer\n    # step, so model.encoder/.classifier\'s actual .data is provably\n    # unchanged from build_specialist_model_cpu\'s own fresh init -- category\n    # 0\'s own loop iteration (reset_specialist_ or\n    # load_specialist_from_checkpoint_, whichever applies) starts from\n    # exactly the state it would have without the smoke tests ever running.\n\n    run_start = time.perf_counter()\n    results = {}\n    stopped = False\n    # REAL AUDIT FINDING (2026-08-22): a real run resuming from\n    # frida-with-merged-lora-and-8-10-14-cat-loras -- which turned out to be\n    # the ORIGINAL pre-NaN-fix run\'s own casualties, not legitimately-cut-\n    # short good checkpoints -- diverged on 8 of the first 10 categories\n    # attempted, even from a fresh init with the already-"fixed"\n    # HEAD_LR_MULTIPLIER/warmup/grad-clip config. That fix has never\n    # actually solved NaN for most categories across MULTIPLE real runs; it\n    # only ever kept 07/10/14 alive. If something is systemically wrong\n    # this run (a bad merged base, a bad data mix, a config regression),\n    # burning the full 8.5h MAX_TRAIN_SECONDS discovering that only at the\n    # very end -- one diverged category at a time -- wastes the entire\n    # session on a foregone conclusion. Stop loudly instead, once enough\n    # CONSECUTIVE categories have diverged that "this one specialist\'s data\n    # is just hard" stops being the more likely explanation.\n    MAX_CONSECUTIVE_CATEGORY_FAILURES = 6\n    consecutive_category_failures = 0\n\n    for cat_id, cat_name in enumerate(category_vocab):\n        if stopped:\n            break\n        label_prefix = f"cat{cat_id:02d}"\n        done_marker = os.path.join(OUTPUT_DIR, f"{label_prefix}_best", "train_state.json")\n        if os.path.isfile(done_marker):\n            _log(rank, "SKIP", f"[{cat_id:02d}] {cat_name}: already finished in an earlier "\n                               f"session -- skipping", force=True)\n            continue\n\n        train_frame, val_frame = train_by_cat[cat_id], val_by_cat[cat_id]\n        n_train = len(train_frame["label"])\n        if n_train < total_batch:\n            # Full-split training (v6) should make this practically\n            # unreachable, but keep v5\'s safety net: a category too small to\n            # form one global batch this run still shouldn\'t lose an\n            # already-validated external checkpoint -- copy it forward\n            # instead of silently dropping it.\n            fallback = _find_external_category_checkpoint(cat_id)\n            if fallback is not None and rank == 0:\n                ext_dir, ext_val = fallback\n                dest_dir = os.path.join(OUTPUT_DIR, f"{label_prefix}_best")\n                _log(rank, "REUSE", f"[{cat_id:02d}] {cat_name}: only {n_train} train rows "\n                                    f"(< one global batch) -- copying forward the existing "\n                                    f"validated checkpoint (val_pr_auc={ext_val:.4f}) instead "\n                                    f"of training: {ext_dir} -> {dest_dir}", force=True)\n                shutil.copytree(ext_dir, dest_dir)\n                results[cat_name] = float(ext_val)\n            elif fallback is None:\n                _log(rank, "SKIP", f"[{cat_id:02d}] {cat_name}: only {n_train} train rows, "\n                                   f"< one global batch, no existing checkpoint to fall back "\n                                   f"to", force=True)\n            continue\n\n        # v6: resume from ANY existing checkpoint (a validated _best, or\n        # failing that a quiet mid-training _last) and CONTINUE training on\n        # the full llm-train split -- per direct instruction ("Инициализация\n        # существующими адаптерами+головами"). v5\'s "already good -> copy\n        # forward, skip entirely" behavior is gone: even a _best checkpoint\n        # only ever saw LLM_ROWS_PER_CATEGORY (100K) rows, so there is real\n        # value left in continuing on the full split regardless of how good\n        # it already is. A category with no checkpoint at all (never reached\n        # by the run being resumed) still falls back to reset_specialist_.\n        external = _find_any_external_category_checkpoint(cat_id)\n        if external is not None:\n            ext_dir, ext_source, ext_val = external\n            try:\n                delta = load_specialist_from_checkpoint_(model, ext_dir)\n                lr_scale = LR_RESUME_MULTIPLIER\n                _log(rank, "RESUME", f"[{cat_id:02d}] {cat_name}: loaded {ext_source} "\n                                     f"checkpoint (val_pr_auc={ext_val}) from {ext_dir} "\n                                     f"(probe delta={delta:.2e}) -- continuing at "\n                                     f"lr_scale={lr_scale}", force=True)\n            except Exception as e:\n                # Real crash: cat00_last (this dataset\'s non-_best categories\n                # are the ORIGINAL pre-NaN-fix run\'s casualties) had already-\n                # corrupted weights, caught by load_specialist_from_\n                # checkpoint_\'s own finiteness check -- but ANY resume\n                # failure (a bad file, a shape mismatch from a differently-\n                # configured checkpoint) deserves the same fallback: a\n                # category that can\'t be resumed is not a category that\n                # should crash the whole 20-category run, it\'s one that\n                # trains from scratch instead.\n                _log(rank, "WARN", f"[{cat_id:02d}] {cat_name}: checkpoint at {ext_dir} "\n                                   f"unusable ({type(e).__name__}: {e}) -- falling back to "\n                                   f"fresh init", force=True)\n                reset_specialist_(model, SEED + cat_id)\n                lr_scale = 1.0\n                external = None\n        else:\n            reset_specialist_(model, SEED + cat_id)\n            lr_scale = 1.0\n            _log(rank, "INIT", f"[{cat_id:02d}] {cat_name}: no existing checkpoint found -- "\n                               f"fresh init, lr_scale={lr_scale}", force=True)\n        model.train()\n\n        # pos_weight from THIS category\'s own mix -- the lever with real\n        # dynamic range (positive rate varies 7.7x across categories, while\n        # category SIZE varies only 1.4x and is therefore not weighted).\n        labels = train_frame["label"]\n        pos_mass = float(labels.sum())\n        pos_weight = torch.tensor(\n            (len(labels) - pos_mass) / max(pos_mass, 1e-9), device=device)\n        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")\n\n        optimizer = torch.optim.AdamW(make_param_groups(model, lr_scale=lr_scale),\n                                      weight_decay=WEIGHT_DECAY)\n        steps_per_epoch = max(1, n_train // total_batch)\n        total_steps = steps_per_epoch * EPOCHS\n        warmup_steps = max(1, int(total_steps * WARMUP_RATIO))\n        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, [\n            _make_lr_lambda(warmup_steps, total_steps, LR_MIN_RATIO),        # adapter\n            _make_head_lr_lambda(warmup_steps, total_steps, LR_MIN_RATIO),   # head\n        ])\n\n        # v9: direct RSS per category, not inferred from wall-clock gaps.\n        # v9\'s own first real run showed inter-category timing growing\n        # (55s -> 73s -> 75s more than the leak-fix-free baseline) alongside\n        # the SAME host-RAM-OOM crash the leak fix was meant to prevent --\n        # consistent with, but not proof of, the extra gc.collect() calls\n        # themselves touching (and possibly COW-duplicating) pages in the\n        # large pre-fork-shared heap under CPython\'s fork+refcount\n        # semantics. This makes that hypothesis checkable directly instead\n        # of inferred a third time from timing.\n        _log(\n            rank,\n            "CAT",\n            f"[{cat_id:02d}] {cat_name}: train={n_train:,} "\n            f"val={len(val_frame[\'label\']):,} steps={total_steps} "\n            f"warmup={warmup_steps} pos_weight={pos_weight.item():.3f} "\n            f"rss_current={_current_rss_mb():.0f}MB "\n            f"rss_peak={_peak_rss_mb():.0f}MB "\n            f"mem_available={_mem_available_mb():.0f}MB",\n            force=True,\n        )\n\n        # V12: do not construct eval loaders until training succeeds.\n        val_full_loader = None\n        val_remainder_loader = None\n        global_step = 0\n        last_quiet_checkpoint = time.perf_counter()\n\n        diverged = False\n        try:\n            for epoch in range(1, EPOCHS + 1):\n                train_loader, sampler = make_train_loader(train_frame, caches, pad_id,\n                                                           PER_CORE_TRAIN_BATCH, world_size,\n                                                           rank, SEED, epoch=epoch)\n                mp_loader = pl_xla.MpDeviceLoader(train_loader, device)\n                for step, batch in enumerate(mp_loader):\n                    input_ids, attention_mask, batch_labels, _cats, batch_scope = batch\n                    batch_labels = batch_labels.to(device)\n                    batch_scope = batch_scope.to(device)\n                    try:\n                        optimizer.zero_grad()\n                        logits = model(input_ids=input_ids, attention_mask=attention_mask)\n                        per_sample = loss_fn(logits, batch_labels)\n                        # human rows count HUMAN_WEIGHT-fold against LLM soft labels\n                        w = torch.where(batch_scope == 0,\n                                        torch.full_like(per_sample, HUMAN_WEIGHT),\n                                        torch.ones_like(per_sample))\n                        loss = (per_sample * w).sum() / w.sum()\n                        # FAIL LOUD, not silently: BCEWithLogitsLoss does not\n                        # raise on NaN, so without this check a diverged run\n                        # would keep computing garbage for the rest of the\n                        # category\'s steps and only surface at eval time. This\n                        # check is cheap (one XLA scalar sync via .item(), same\n                        # cost class as the existing loss.item() logging below)\n                        # and catches it the moment it happens instead.\n                        if not torch.isfinite(loss).item():\n                            raise TrainingDivergedError(\n                                f"[{cat_id:02d}] {cat_name}: loss is {loss.item()} (non-finite) "\n                                f"at step {global_step + 1}/{total_steps}")\n                        loss.backward()\n                        # Added after the same real NaN: clips across BOTH param\n                        # groups together (one global norm), matching\n                        # clip_grad_norm_\'s own default behavior.\n                        torch.nn.utils.clip_grad_norm_(\n                            [p for g in optimizer.param_groups for p in g["params"]],\n                            GRAD_CLIP_NORM)\n                        xm.optimizer_step(optimizer)\n                        scheduler.step()\n                        global_step += 1\n                    except TrainingDivergedError:\n                        # Let this one through -- caught at the CATEGORY level\n                        # below, not swallowed here as "just another skipped\n                        # step". Cleans up the same way a skipped step would.\n                        try:\n                            del logits, per_sample, loss\n                        except NameError:\n                            pass\n                        optimizer.zero_grad(set_to_none=True)\n                        raise\n                    except Exception as e:\n                        _log(rank, "WARN", f"step failed and was skipped: {type(e).__name__}: {e}",\n                             force=True)\n                        # del before continue: a live reference to the failed step\'s\n                        # graph keeps its buffers alive into the next step.\n                        try:\n                            del logits, per_sample, loss\n                        except NameError:\n                            pass\n                        optimizer.zero_grad(set_to_none=True)\n                        continue\n\n                    if global_step % LOG_EVERY_STEPS == 0:\n                        xm.mark_step()\n                        if rank == 0:\n                            print(f"[{_elapsed():8.1f}s] cat={cat_id:02d} step={global_step}/"\n                                  f"{total_steps} loss={loss.item():.4f} "\n                                  f"lr={scheduler.get_last_lr()[0]:.2e}/"\n                                  f"{scheduler.get_last_lr()[1]:.2e}", flush=True)\n\n                    # v6: quiet checkpoints -- adapter + head only (no eval, no\n                    # val_pr_auc), so a session killed mid-category loses at most\n                    # CHECKPOINT_EVERY_SECONDS of progress instead of the whole\n                    # category. Saved under the same "_last" label a NaN-failure\n                    # already used, since both mean the same thing to a resumer:\n                    # "unvalidated, but has real trained weights."\n                    if (time.perf_counter() - last_quiet_checkpoint) > CHECKPOINT_EVERY_SECONDS:\n                        xm.mark_step()\n                        save_checkpoint(model, optimizer, global_step, epoch, None,\n                                        f"{label_prefix}_last", rank,\n                                        extra={"category": cat_name, "category_id": cat_id,\n                                               "n_train": int(n_train), "quiet_checkpoint": True})\n                        last_quiet_checkpoint = time.perf_counter()\n                        _log(rank, "CKPT", f"[{cat_id:02d}] {cat_name}: quiet checkpoint "\n                                           f"(every {CHECKPOINT_EVERY_SECONDS}s) at "\n                                           f"step={global_step}/{total_steps}")\n\n                    if (time.perf_counter() - run_start) > MAX_TRAIN_SECONDS:\n                        _log(rank, "TRAIN", f"wall-clock cutoff during category {cat_id:02d} "\n                                            f"-- stopping. Finished categories keep their "\n                                            f"checkpoints; a re-run resumes from here.", force=True)\n                        stopped = True\n                        break\n                del mp_loader, train_loader, sampler\n                if stopped:\n                    break\n        except TrainingDivergedError as e:\n            _log(rank, "ERROR", f"[{cat_id:02d}] {cat_name}: training diverged -- {e} -- "\n                                f"skipping eval and moving to the next category instead of "\n                                f"crashing the whole run.", force=True)\n            diverged = True\n            # REAL CRASH (host RAM OOM, v8): a raise from inside the step loop\n            # jumps straight here, OVER the `del mp_loader, train_loader,\n            # sampler` a normal epoch end reaches (a few lines up) -- before\n            # v8 this never mattered (divergence used to crash the whole\n            # process immediately, so nothing ever accumulated across\n            # categories). Now that divergence is the common case (8/10\n            # categories in the real run that found this), the abandoned\n            # DataLoaders\' num_workers=2 subprocesses piled up across rapid\n            # category transitions (several categories diverged within 2-5\n            # steps) until the host ran out of RAM -- confirmed by the\n            # "Your notebook tried to allocate more memory than is\n            # available" kill and by rank divergence timing spreading to 28s\n            # (vs <1s normally) right before it. Same cleanup as the normal\n            # path, just reachable from here too. Unconditional del is safe:\n            # TrainingDivergedError can only be raised from inside the step\n            # loop, which never runs before these three are assigned for the\n            # current epoch. No gc.collect() here -- del alone already drops\n            # these to refcount 0 (non-cyclic), and the ONE combined collect\n            # below (after val loaders too) covers cycles for the whole\n            # category in a single pass instead of three separate full-heap\n            # scans stacking up per category (v9\'s own real run showed\n            # per-category wall-clock growing 55-96s more than a run without\n            # the extra collect() calls -- consistent with each full GC pass\n            # over a large pre-fork-shared heap costing real, and possibly\n            # COW-duplicating, time under CPython\'s fork+refcount semantics).\n            del mp_loader, train_loader, sampler\n\n        if diverged:\n            xm.mark_step()\n            save_checkpoint(model, optimizer, global_step, EPOCHS, None,\n                            f"{label_prefix}_last", rank,\n                            extra={"category": cat_name, "category_id": cat_id,\n                                   "resumed_from": external[1] if external is not None else None,\n                                   "lr_scale": lr_scale, "diverged": True})\n            results[cat_name] = None\n        else:\n            try:\n                val_full_loader, val_remainder_loader = make_eval_loader(\n                    val_frame, caches, pad_id, PER_CORE_EVAL_BATCH\n                )\n                m = evaluate(model, val_full_loader, val_remainder_loader, device, category_vocab)\n                pr_auc = m["per_category"].get(cat_name, float("nan"))\n                _log(rank, "METRICS", f"[{cat_id:02d}] {cat_name}: val_pr_auc={pr_auc:.4f} "\n                                      f"n_val={m[\'n\']}", force=True)\n                save_checkpoint(model, optimizer, global_step, EPOCHS, float(pr_auc),\n                                f"{label_prefix}_best", rank,\n                                extra={"category": cat_name, "category_id": cat_id,\n                                       "n_train": int(n_train),\n                                       "resumed_from": external[1] if external is not None else None,\n                                       "lr_scale": lr_scale,\n                                       "llm_is_subsample": False})\n                results[cat_name] = float(pr_auc)\n            except Exception as e:\n                _log(rank, "ERROR", f"[{cat_id:02d}] eval failed: {type(e).__name__}: {e}", force=True)\n                save_checkpoint(model, optimizer, global_step, EPOCHS, None,\n                                f"{label_prefix}_last", rank,\n                                extra={"category": cat_name, "category_id": cat_id,\n                                       "resumed_from": external[1] if external is not None else None,\n                                       "lr_scale": lr_scale,\n                                       "llm_is_subsample": False})\n                results[cat_name] = None\n        # REAL CRASH (host RAM OOM, v8): val_full_loader/val_remainder_loader\n        # (num_workers=2 each) are built fresh every category -- ALL 20 now,\n        # not just the ones that needed real training (v8 dropped v5\'s\n        # skip-if-externally-good `continue`, which used to return before\n        # these were even constructed) -- and were never explicitly freed,\n        # relying only on the implicit refcount drop when the names get\n        # reassigned at the top of the NEXT category\'s loop. Explicit here\n        # for the same reason as the train-loader del above: force the\n        # worker subprocesses to tear down now, not whenever gc gets to it.\n        # ONE gc.collect() here covers both this del and the diverged-path\n        # train-loader del above (which deliberately skips its own call) --\n        # cycles from the whole category collected in a single pass.\n        del val_full_loader, val_remainder_loader\n        del optimizer, scheduler, loss_fn, pos_weight\n        # Refcount cleanup is enough here; avoid full-heap GC in forked ranks.\n        model.train()\n\n        if diverged:\n            consecutive_category_failures += 1\n        else:\n            consecutive_category_failures = 0\n        if consecutive_category_failures >= MAX_CONSECUTIVE_CATEGORY_FAILURES:\n            raise RuntimeError(\n                f"[rank{rank}] {consecutive_category_failures} categories in a row diverged to "\n                f"NaN ({cat_id - consecutive_category_failures + 1:02d}..{cat_id:02d}) -- this is "\n                f"not \'{cat_name}\'s own data is hard\', it is a systemic problem (merged base, "\n                f"data mix, or config) common to all of them. Stopping now instead of burning "\n                f"MAX_TRAIN_SECONDS to reach the same conclusion 14 categories later. Checkpoints "\n                f"already saved for finished/attempted categories are untouched -- a re-run "\n                f"resumes from here once the systemic cause is fixed.")\n\n        # Mirror results to disk after EVERY category: a cancelled session can\n        # come back with a 0-byte log, and this is then the only record.\n        if rank == 0:\n            done = [v for v in results.values() if v is not None]\n            with open(RESULTS_JSON, "w", encoding="utf-8") as f:\n                json.dump({"per_category_val_pr_auc": results,\n                           "macro_over_finished": float(np.mean(done)) if done else None,\n                           "n_finished": len(done),\n                           "n_categories": len(category_vocab),\n                           "llm_is_subsample": False,\n                           "llm_split_path_hint": LLM_SPLIT_PATH_HINT,\n                           "llm_rows_total_in_split": _SHARED["n_llm_total"],\n                           "llm_rows_used_train": _SHARED["n_llm_used"],\n                           "pair_max_length": PAIR_MAX_LENGTH,\n                           "val_fingerprint": _SHARED["val_fingerprint"],\n                           "split": "OLD canonical pair-level 80/20 seed42"}, f,\n                          ensure_ascii=False, indent=2)\n\n    done = [v for v in results.values() if v is not None]\n    _log(rank, "DONE", f"total wall-clock {_elapsed():.0f}s  finished={len(done)}/"\n                       f"{len(category_vocab)} categories  "\n                       f"macro_over_finished={np.mean(done) if done else float(\'nan\'):.4f}",\n         force=True)\n\n\ndef main():\n    global _WRAPPED_MODEL\n    os.environ.pop("TPU_PROCESS_ADDRESSES", None)\n    os.environ.pop("CLOUD_TPU_TASK_ID", None)\n\n    _load_shared_data()\n\n    # V12: return allocator slack before building the model and before fork.\n    gc.collect()\n    _malloc_trim()\n    _mem_log("after data setup + malloc_trim, before model build")\n\n    # v11: build ONCE here, pre-fork, and wrap -- see\n    # build_specialist_model_cpu\'s docstring (REAL CRASH, host RAM OOM) for\n    # why this moved out of _mp_fn. MpModelWrapper\'s whole point is that the\n    # wrap must happen BEFORE the fork inside torch_xla.launch() below, so\n    # every rank inherits these host pages via copy-on-write instead of\n    # each one loading its own.\n    model_cpu, hidden_dim, n_trainable = build_specialist_model_cpu(_SHARED["merged_dir"])\n    _SHARED["n_trainable"] = n_trainable\n    _WRAPPED_MODEL = xmp.MpModelWrapper(model_cpu)\n    print(f"[{_elapsed():8.1f}s][MODEL] built once pre-fork, wrapped for "\n          f"copy-on-write sharing across ranks (hidden_dim={hidden_dim}, "\n          f"trainable={n_trainable:,})", flush=True)\n\n    gc.collect()\n    _malloc_trim()\n    _mem_log("after model build, immediately before fork")\n\n    # Python pre-fork COW recipe: keep the inherited long-lived object graph\n    # out of child cyclic-GC scans.\n    gc.disable()\n    if hasattr(gc, "freeze"):\n        gc.freeze()\n\n    torch_xla.launch(_mp_fn, args=(), start_method="fork")\n\n\nif __name__ == "__main__":\n    main()\n'

SCRIPT.write_text(PATCHED_SOURCE, encoding="utf-8")

print("Wrote:", SCRIPT)
print("Lines:", len(PATCHED_SOURCE.splitlines()))

In [ ]:
# Syntax check before spending TPU time.

import sys, subprocess

subprocess.run(
    [sys.executable, "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("Syntax: OK")

## Run

Лучше запускать сам training script в **fresh subprocess** с ограниченным числом
glibc arenas. Это помогает именно при fork-heavy workloads и снижает fragmentation/COW.

Нужны те же Kaggle Inputs, что и у предыдущей версии:
- FRIDA general adapter / merged-base source;
- human + LLM data;
- v2/v3 token caches;
- LLM train/val split;
- при resume — dataset с предыдущими category checkpoints.

In [ ]:
import os, sys, subprocess, time

env = os.environ.copy()

# Must be present BEFORE the fresh Python process starts.
env["MALLOC_ARENA_MAX"] = "2"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

print("Launching:", SCRIPT)
print("MALLOC_ARENA_MAX =", env["MALLOC_ARENA_MAX"])

t0 = time.time()

res = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)],
    env=env,
)

print("\nReturn code:", res.returncode)
print("Wall hours:", (time.time() - t0) / 3600)

if res.returncode != 0:
    print(
        "\nIf the tail again contains only SIGTERM/BrokenProcessPool, inspect the last "
        "MEM/CAT line. The important fields are rss_current and mem_available, not rss_peak."
    )

assert res.returncode == 0, f"Training subprocess exited with code {res.returncode}"

## Что смотреть в следующем логе

Для диагностики памяти теперь будут строки примерно такого вида:

```text
[MEM] after data setup + malloc_trim, before model build:
rss_current=...MB rss_peak=...MB mem_available=...MB

[MEM] after model build, immediately before fork:
rss_current=...MB rss_peak=...MB mem_available=...MB

[rankN] MEM: after model.to(device):
rss_current=...MB rss_peak=...MB mem_available=...MB

[rankN] CAT:
... rss_current=...MB rss_peak=...MB mem_available=...MB
```

Если опять произойдёт внешний `SIGTERM`, этих четырёх точек уже хватит, чтобы понять,
**где именно появляется RAM pressure**: data setup, model fork/device transfer или
реальный training batch.

`rss_peak` может оставаться большим после временного preprocessing allocation — это
нормально. Для текущего состояния процесса важнее `rss_current` и `MemAvailable`.